# __선형회귀 (Linear Regression)__
### 이론부터 WM-811K 웨이퍼 맵 회귀분석까지

<a href="https://colab.research.google.com/github/Jungsu-Yun/CNUML/blob/main/%EC%84%A0%ED%98%95%ED%9A%8C%EA%B7%80_WM811K.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

선형회귀를 **수학적으로 유도**하고, **PyTorch로 직접 구현**한 다음, 실제 반도체 공정 데이터인 **WM-811K 웨이퍼 맵(81만 장)** 으로 회귀분석을 처음부터 끝까지 해 봐요.

**학습 목표**
1. 최소제곱 문제를 **정규방정식 · 사영 · 최대우도** 세 관점에서 이해하고, 추정량의 분포(추론)까지 유도할 수 있다.
2. 같은 모델을 **닫힌 해(QR / `lstsq`)** 와 **경사하강법(autograd, `nn.Module`, `DataLoader`)** 으로 PyTorch에서 학습할 수 있다.
3. **기저함수 확장 · 과적합 · 릿지 정규화**를 교차검증으로 다룰 수 있다.
4. 실제 데이터를 탐색하고, 문제를 **회귀 문제로 정식화**해 학습 · 평가 · 시각화할 수 있다.

**표기법**

| 기호 | 의미 |
|---|---|
| $n$ | 관측(샘플) 수 |
| $p$ | 절편을 포함한 회귀계수 개수 |
| $X\in\mathbb{R}^{n\times p}$ | 설계행렬 (design matrix), 첫 열은 $\mathbf 1$ |
| $\mathbf y\in\mathbb{R}^{n}$ | 타깃 벡터 |
| $\boldsymbol\beta\in\mathbb{R}^{p}$ | 회귀계수 |
| $\boldsymbol\varepsilon$ | 오차항, $\mathbb E[\boldsymbol\varepsilon]=\mathbf 0$, $\operatorname{Cov}(\boldsymbol\varepsilon)=\sigma^2 I_n$ |
| $H = X(X^\top X)^{-1}X^\top$ | hat matrix (열공간으로의 사영) |
| $\hat{\ }$ (hat) | 데이터로부터 추정한 값 |

---
# __PART 0. 환경 설정__

### 0-1. 의존성 설치

| 패키지 | 용도 | 확인한 버전 |
|---|---|---|
| `torch` | 텐서 연산 · 선형대수 · 자동미분 · 모델 학습 · 통계량 계산 | 2.4 ~ 2.15 |
| `pandas` | 데이터프레임 다루기 · 그룹 집계(`groupby`) · 표 출력 | 2.2 ~ 3.1 |
| `numpy` | pickle 안의 배열 복원 (pandas · torch의 의존성) | 1.26 ~ 2.5 |
| `matplotlib` | 시각화 | 3.8 ~ 3.11 |
| `kagglehub` | Kaggle 데이터셋 내려받기 (로그인 불필요) | 0.3 ~ 1.0 |

Colab에는 필요한 패키지가 모두 기본 설치되어 있어서, 아래 셀은 Colab에서는 **`kagglehub`만 확인하고 금방 끝나요**.
Colab의 `torch`는 CUDA 라이브러리(`nvidia-*`)와 짝을 맞춰 설치되어 있어요. pip에 `torch`를 다시 넘기면 이 라이브러리들을 수 GB 새로 받는 경우가 있어서 건드리지 않아요.
실행 후 `Note: you may need to restart the kernel ...` 이 보이면 pip의 기본 안내 문구예요. **새로 설치된 패키지가 있을 때만** 메뉴 **런타임 → 세션 다시 시작** 후 다음 셀부터 실행하세요.

<details>
<summary>로컬(내 컴퓨터)에서 실행할 때</summary>

```bash
python -m venv .venv && source .venv/bin/activate      # Windows: .venv\Scripts\activate
pip install "torch>=2.4" "numpy>=1.26" "pandas>=2.2" "matplotlib>=3.8" "kagglehub>=0.3" jupyter
jupyter notebook
```
Linux에서는 `pip install torch` 가 기본으로 CUDA 빌드(수 GB)를 받아요. GPU가 없거나 Windows에서 CUDA용이 필요하면 https://pytorch.org/get-started/locally/ 에서 맞는 명령을 고르세요. 메모리는 **8GB 이상**을 권장해요 (원본 데이터가 메모리에서 약 3GB).
</details>

In [ ]:
import sys
if "google.colab" in sys.modules:
    # Colab: torch · numpy · pandas · matplotlib은 이미 설치되어 있어요 → kagglehub만 확인
    print("Colab 환경: kagglehub만 확인해요")
    %pip install -q "kagglehub>=0.3"
else:
    # 로컬: 확인한 버전 이상으로 설치 (이미 만족하면 다시 설치하지 않아요. 처음 설치라면 몇 분 걸려요)
    print("로컬 환경: torch · numpy · pandas · matplotlib · kagglehub를 확인/설치해요")
    %pip install -q "torch>=2.4" "numpy>=1.26" "pandas>=2.2" "matplotlib>=3.8" "kagglehub>=0.3"
print("완료")

### 0-2. 라이브러리 불러오기
모델링 · 선형대수 · 학습 · 통계량 계산은 모두 `torch`로 해요. `pandas`는 표 정리와 그룹별 합계 · 평균 같은 집계(`groupby`)에만, `numpy`는 pickle 복원과 그림을 그릴 때 텐서를 넘기는 데에만 써요.
닫힌 해와 통계량은 수치 오차를 줄이려고 **배정밀도(`float64`)** 로, 신경망 방식 학습은 `float32`로 계산해요.

In [ ]:
import math
import pickle
import time
from pathlib import Path

import numpy as np          # pickle 안의 배열 복원용 (계산은 torch로 해요)
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import kagglehub

SEED = 0
torch.manual_seed(SEED)
F64 = torch.float64                                   # 닫힌 해 · 통계 계산용 dtype
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.precision", 4)

for name, mod in [("torch", torch), ("numpy", np), ("pandas", pd),
                  ("matplotlib", matplotlib), ("kagglehub", kagglehub)]:
    print(f"{name:11s} {mod.__version__}")
print("device     ", device)

### 0-3. WM-811K 데이터 내려받기
**지금 바로 실행해 두세요.** PART 1을 설명하는 동안 다운로드가 끝나요. (압축 149MB → 압축 해제 후 `LSWMD.pkl` 약 2.1GB, Colab에서 약 1분)

> 다운로드가 실패하면
> 1. Kaggle에 로그인해 API 토큰을 받은 뒤 Colab 왼쪽 열쇠 아이콘(보안 비밀)에 `KAGGLE_USERNAME`, `KAGGLE_KEY` 를 저장하거나, 새 셀에서 `kagglehub.login()` 을 실행하고 다시 시도하세요.
> 2. 또는 [Kaggle 데이터셋 페이지](https://www.kaggle.com/datasets/qingyi/wm811k-wafer-map)에서 압축 파일(약 149MB)을 받아 Google Drive에 올린 뒤, 아래 셀 대신 새 셀에서 Colab 디스크에 풀어서 쓰세요.
>    ```python
>    from google.colab import drive
>    drive.mount("/content/drive")
>    !unzip -q -o "/content/drive/MyDrive/archive.zip" -d /content/wm811k   # 올린 파일 경로에 맞게 바꾸세요
>    PKL_PATH = Path("/content/wm811k/LSWMD.pkl")
>    ```

In [ ]:
data_dir = Path(kagglehub.dataset_download("qingyi/wm811k-wafer-map"))
PKL_PATH = data_dir / "LSWMD.pkl"
print(PKL_PATH, f"({PKL_PATH.stat().st_size / 1e9:.2f} GB)")

---
# PART 1. 선형회귀의 수학과 PyTorch 구현

### 1-1. 문제 설정
관측 $(\mathbf x_i, y_i)$, $i=1,\dots,n$, $\mathbf x_i\in\mathbb R^{p-1}$ 에 대해 선형회귀 모형은

$$y_i = \beta_0 + \beta_1 x_{i1} + \cdots + \beta_{p-1}x_{i,p-1} + \varepsilon_i
\qquad\Longleftrightarrow\qquad
\mathbf y = X\boldsymbol\beta + \boldsymbol\varepsilon,\quad
X=\begin{bmatrix}1 & \mathbf x_1^\top\\ \vdots & \vdots\\ 1 & \mathbf x_n^\top\end{bmatrix}.$$

**표준 가정**
1. 선형성: $\mathbb E[\mathbf y\mid X]=X\boldsymbol\beta$
2. 완전 열계수: $\operatorname{rank}(X)=p$
3. 외생성: $\mathbb E[\boldsymbol\varepsilon\mid X]=\mathbf 0$
4. 등분산 · 무상관: $\operatorname{Cov}(\boldsymbol\varepsilon\mid X)=\sigma^2 I_n$
5. (추론용) 정규성: $\boldsymbol\varepsilon\mid X\sim\mathcal N(\mathbf 0,\sigma^2 I_n)$

> "선형"은 **파라미터 $\boldsymbol\beta$ 에 대해 선형**이라는 뜻이에요. $x^2$, $\sin x$, $\log x$ 같은 변환을 특성으로 써도 여전히 선형회귀예요 (1-7, PART 2-7, PART 4).

### 1-2. 예제 데이터
참값을 아는 합성 데이터 $y = 2 + 0.5x + \varepsilon$, $\varepsilon\sim\mathcal N(0,1)$ 로 시작해요.

In [ ]:
g = torch.Generator().manual_seed(42)
n, sigma = 200, 1.0
beta_true = torch.tensor([2.0, 0.5], dtype=F64)          # (β0, β1)

x = 10 * torch.rand(n, generator=g, dtype=F64)            # x ~ U(0, 10)
X = torch.stack([torch.ones(n, dtype=F64), x], dim=1)     # 설계행렬 (n, p=2)
y = X @ beta_true + sigma * torch.randn(n, generator=g, dtype=F64)

xx = torch.linspace(0, 10, 100, dtype=F64)
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.scatter(x.numpy(), y.numpy(), s=10, alpha=0.6, label="data")
ax.plot(xx.numpy(), (beta_true[0] + beta_true[1] * xx).numpy(), "k--", label="true line")
ax.set(xlabel="x", ylabel="y", title=r"Synthetic data: $y = 2 + 0.5x + \varepsilon$")
ax.legend()
plt.show()

### 1-3. 최소제곱 추정과 정규방정식
잔차제곱합(RSS)을 최소화해요.

$$L(\boldsymbol\beta)=\lVert\mathbf y-X\boldsymbol\beta\rVert_2^2=\mathbf y^\top\mathbf y-2\boldsymbol\beta^\top X^\top\mathbf y+\boldsymbol\beta^\top X^\top X\boldsymbol\beta$$

$$\nabla_{\boldsymbol\beta}L=-2X^\top\mathbf y+2X^\top X\boldsymbol\beta=\mathbf 0
\quad\Longrightarrow\quad
\boxed{X^\top X\,\hat{\boldsymbol\beta}=X^\top\mathbf y}\ \ (\text{정규방정식})$$

- 헤시안 $\nabla^2L=2X^\top X\succeq 0$ → $L$은 볼록 함수 → 정상점이 곧 전역 최솟점이에요.
- $\operatorname{rank}(X)=p$ 이면 $X^\top X\succ0$ → 해가 유일: $\hat{\boldsymbol\beta}=(X^\top X)^{-1}X^\top\mathbf y$.

**수치적으로는** 계수를 구할 때 $(X^\top X)^{-1}$ 을 직접 계산하지 않아요 (표준오차처럼 $(X^\top X)^{-1}$ 자체가 필요할 때만 계산해요). $\kappa(X^\top X)=\kappa(X)^2$ 이라 조건수가 **제곱**으로 나빠지기 때문이에요. 대신 QR 분해 $X=QR$ ($Q^\top Q=I_p$, $R$ 상삼각)을 쓰면

$$X^\top X\boldsymbol\beta=X^\top\mathbf y\iff R^\top R\boldsymbol\beta=R^\top Q^\top\mathbf y\iff R\boldsymbol\beta=Q^\top\mathbf y$$

조건수가 $\kappa(X)$ 인 삼각 연립방정식만 풀면 돼요. `torch.linalg.lstsq`도 직교분해 기반(LAPACK) 풀이를 써요.

In [ ]:
beta_ne = torch.linalg.solve(X.T @ X, X.T @ y)                               # ① 정규방정식
Q, R = torch.linalg.qr(X)                                                    # ② QR: X = QR
beta_qr = torch.linalg.solve_triangular(R, (Q.T @ y).unsqueeze(1), upper=True).squeeze(1)
beta_ls = torch.linalg.lstsq(X, y.unsqueeze(1)).solution.squeeze(1)          # ③ lstsq

for name, b in [("normal eq.", beta_ne), ("QR", beta_qr), ("lstsq", beta_ls), ("true", beta_true)]:
    print(f"{name:10s}  β0 = {b[0].item():.6f}   β1 = {b[1].item():.6f}")

kX = torch.linalg.cond(X).item()
print(f"\ncond(X) = {kX:.1f},  cond(XᵀX) = {torch.linalg.cond(X.T @ X).item():.1f}  (cond(X)² = {kX**2:.1f})")

### 1-4. 기하학적 해석: 열공간으로의 직교사영

$$\hat{\mathbf y}=X\hat{\boldsymbol\beta}=\underbrace{X(X^\top X)^{-1}X^\top}_{H}\,\mathbf y$$

- $H^\top=H$, $H^2=H$ → $H$는 $\operatorname{col}(X)$ 위로의 **직교사영행렬**
- $\operatorname{tr}(H)=\operatorname{tr}\big((X^\top X)^{-1}X^\top X\big)=p$ → 모형의 자유도
- 잔차 $\mathbf e=(I-H)\mathbf y$ 는 열공간과 직교: $X^\top\mathbf e=\mathbf 0$ ← **정규방정식 그 자체!**
- 절편이 있으면 $\mathbf 1\in\operatorname{col}(X)$ 이라서 $\bar{\hat y}=\bar y$, 그리고 피타고라스 정리가 성립해요.

$$\underbrace{\lVert\mathbf y-\bar y\mathbf 1\rVert^2}_{\text{TSS}}=\underbrace{\lVert\hat{\mathbf y}-\bar y\mathbf 1\rVert^2}_{\text{ESS}}+\underbrace{\lVert\mathbf e\rVert^2}_{\text{RSS}},
\qquad R^2=1-\frac{\text{RSS}}{\text{TSS}}=\frac{\text{ESS}}{\text{TSS}}\in[0,1]$$

평가 지표 함수도 여기서 정의해 두고 끝까지 써요.

In [ ]:
def r2_score(y, y_hat):
    """결정계수 R² = 1 − RSS/TSS"""
    return 1 - ((y - y_hat) ** 2).sum() / ((y - y.mean()) ** 2).sum()

def rmse(y, y_hat):
    return torch.sqrt(((y - y_hat) ** 2).mean())

H = X @ torch.linalg.solve(X.T @ X, X.T)            # hat matrix (n × n)
y_hat = H @ y
e = y - y_hat
tss, ess, rss = ((y - y.mean())**2).sum(), ((y_hat - y.mean())**2).sum(), (e**2).sum()

print(f"‖Hᵀ − H‖_F = {torch.linalg.matrix_norm(H.T - H).item():.2e}")
print(f"‖H² − H‖_F = {torch.linalg.matrix_norm(H @ H - H).item():.2e}")
print(f"tr(H)      = {torch.trace(H).item():.6f}   (p = {X.shape[1]})")
print(f"Xᵀe        = {(X.T @ e).numpy()}")
print(f"TSS = {tss.item():.4f}   ESS + RSS = {(ess + rss).item():.4f}")
print(f"R² = 1 − RSS/TSS = {(1 - rss / tss).item():.4f}   ESS/TSS = {(ess / tss).item():.4f}   r2_score = {r2_score(y, y_hat).item():.4f}")

> 💬 **생각해 볼 질문**: 절편이 없는 모형($\mathbf 1\notin\operatorname{col}(X)$)에서도 $1-\text{RSS}/\text{TSS}=\text{ESS}/\text{TSS}$ 가 성립할까요? $R^2<0$ 이 나올 수 있을까요?

### 1-5. 확률적 관점: 최대우도와 추정량의 분포
$\boldsymbol\varepsilon\sim\mathcal N(\mathbf 0,\sigma^2 I)$ 이면 로그우도는

$$\ell(\boldsymbol\beta,\sigma^2)=-\frac n2\log(2\pi\sigma^2)-\frac1{2\sigma^2}\lVert\mathbf y-X\boldsymbol\beta\rVert^2$$

이므로 $\boldsymbol\beta$ 의 **MLE = 최소제곱해**예요. ($\sigma^2$ 를 고정하면 음의 로그우도 $=\frac{n}{2\sigma^2}\,\text{MSE}+\text{상수}$ 라서, MSE 최소화와 같아요.)

$\hat{\boldsymbol\beta}=(X^\top X)^{-1}X^\top(X\boldsymbol\beta+\boldsymbol\varepsilon)=\boldsymbol\beta+(X^\top X)^{-1}X^\top\boldsymbol\varepsilon$ 에서

$$\mathbb E[\hat{\boldsymbol\beta}]=\boldsymbol\beta,\qquad
\operatorname{Cov}(\hat{\boldsymbol\beta})=\sigma^2(X^\top X)^{-1},\qquad
\hat{\boldsymbol\beta}\sim\mathcal N\big(\boldsymbol\beta,\ \sigma^2(X^\top X)^{-1}\big).$$

- $\mathbb E[\text{RSS}]=\mathbb E[\boldsymbol\varepsilon^\top(I-H)\boldsymbol\varepsilon]=\sigma^2\operatorname{tr}(I-H)=\sigma^2(n-p)$ → 불편추정량 $s^2=\text{RSS}/(n-p)$ (MLE $\text{RSS}/n$ 은 편향)
- 표준오차 $\operatorname{se}(\hat\beta_j)=s\sqrt{[(X^\top X)^{-1}]_{jj}}$, 검정통계량 $t_j=\hat\beta_j/\operatorname{se}(\hat\beta_j)\sim t_{n-p}$ (귀무가설 $\beta_j=0$)
- **Gauss–Markov 정리**: 정규성 없이 가정 1–4만으로도 OLS는 선형 불편추정량 중 분산이 가장 작아요 (BLUE).

아래 `ols` 함수를 PART 2, 3에서도 그대로 써요. (`torch`에는 $t$ 분포의 분위수 함수가 없어서 정규근사 $z_{0.975}=1.96$ 을 써요. $n-p=198$ 이면 $t_{0.975}\approx1.972$ 라 차이가 아주 작아요.)

In [ ]:
def ols(X, y):
    """X: 절편 열을 포함한 설계행렬. 계수 · 표준오차 · t값 · R² 를 돌려줘요."""
    n, p = X.shape
    beta = torch.linalg.lstsq(X, y.unsqueeze(1)).solution.squeeze(1)
    resid = y - X @ beta
    s2 = (resid ** 2).sum() / (n - p)                 # σ² 의 불편추정량
    cov = s2 * torch.linalg.inv(X.T @ X)              # Cov(β̂) 추정
    se = torch.sqrt(torch.diag(cov))
    return {"beta": beta, "se": se, "t": beta / se, "s2": s2, "cov": cov, "r2": r2_score(y, X @ beta)}

Z975 = torch.distributions.Normal(0.0, 1.0).icdf(torch.tensor(0.975)).item()   # 1.95996…

fit = ols(X, y)
pval = 2 * (1 - torch.special.ndtr(fit["t"].abs()))   # 정규근사 양측 p-value
pd.DataFrame({"true": beta_true.numpy(), "estimate": fit["beta"].numpy(), "se": fit["se"].numpy(),
              "t": fit["t"].numpy(), "p (normal)": pval.numpy(),
              "95% CI low": (fit["beta"] - Z975 * fit["se"]).numpy(),
              "95% CI high": (fit["beta"] + Z975 * fit["se"]).numpy()}, index=["β0", "β1"])

#### (심화) 몬테카를로로 확인하기
같은 $X$ 에 잡음만 새로 뽑아 실험을 5,000번 반복해요. `torch.linalg.lstsq`는 우변이 행렬이면 **열마다 독립된 최소제곱 문제**를 한 번에 풀어요.

In [ ]:
M = 5000
g = torch.Generator().manual_seed(7)
Y = (X @ beta_true).unsqueeze(1) + sigma * torch.randn(n, M, generator=g, dtype=F64)   # (n, M)
B = torch.linalg.lstsq(X, Y).solution                       # (p, M): β̂ 5,000개
XtX_inv = torch.linalg.inv(X.T @ X)

print("경험적 Cov(β̂):\n", torch.cov(B).numpy())
print("이론 σ²(XᵀX)⁻¹:\n", (sigma**2 * XtX_inv).numpy())

s2_mc = ((Y - X @ B) ** 2).sum(0) / (n - 2)
se1 = torch.sqrt(s2_mc * XtX_inv[1, 1])
covered = ((B[1] - Z975 * se1 <= beta_true[1]) & (beta_true[1] <= B[1] + Z975 * se1)).double().mean()
print(f"\nβ1 의 95% 신뢰구간이 참값을 포함한 비율: {covered.item():.3f}")

sd1 = math.sqrt(sigma**2 * XtX_inv[1, 1].item())
grid = torch.linspace(B[1].min().item(), B[1].max().item(), 200, dtype=F64)
pdf = torch.exp(torch.distributions.Normal(beta_true[1], sd1).log_prob(grid))
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.hist(B[1].numpy(), bins=60, density=True, alpha=0.6, label=r"Monte Carlo $\hat\beta_1$")
ax.plot(grid.numpy(), pdf.numpy(), "r-", label=r"$\mathcal{N}(\beta_1, \sigma^2[(X^TX)^{-1}]_{11})$")
ax.set(xlabel=r"$\hat\beta_1$", ylabel="density", title="Sampling distribution of the OLS slope")
ax.legend()
plt.show()

### 1-6. 경사하강법과 조건수
평균제곱오차 $J(\boldsymbol\beta)=\frac1n\lVert\mathbf y-X\boldsymbol\beta\rVert^2$ 의 기울기와 헤시안은

$$\nabla J(\boldsymbol\beta)=\tfrac2n X^\top(X\boldsymbol\beta-\mathbf y),\qquad A:=\nabla^2J=\tfrac2nX^\top X\ \ (\text{상수 행렬}).$$

갱신식 $\boldsymbol\beta_{t+1}=\boldsymbol\beta_t-\eta\nabla J(\boldsymbol\beta_t)$ 와 $\nabla J(\hat{\boldsymbol\beta})=\mathbf 0$ 에서 **오차가 선형 점화식**을 따라요.

$$\boldsymbol\beta_{t+1}-\hat{\boldsymbol\beta}=(I-\eta A)(\boldsymbol\beta_t-\hat{\boldsymbol\beta})
\quad\Longrightarrow\quad
\boldsymbol\beta_t-\hat{\boldsymbol\beta}=(I-\eta A)^t(\boldsymbol\beta_0-\hat{\boldsymbol\beta}).$$

$A$ 의 고윳값을 $0<\lambda_{\min}\le\cdots\le\lambda_{\max}$ 라 하면
- **수렴 조건**: $\rho(I-\eta A)=\max_i|1-\eta\lambda_i|<1\iff 0<\eta<2/\lambda_{\max}$
- 최적 학습률 $\eta^\star=2/(\lambda_{\max}+\lambda_{\min})$ 에서 수렴률 $\rho^\star=\dfrac{\kappa-1}{\kappa+1}$, $\kappa=\lambda_{\max}/\lambda_{\min}$
  (이건 $\lVert\boldsymbol\beta_t-\hat{\boldsymbol\beta}\rVert$ 의 수렴률이에요. 아래 오른쪽 그림의 $J(\boldsymbol\beta_t)-J^\star=\tfrac12(\boldsymbol\beta_t-\hat{\boldsymbol\beta})^\top A(\boldsymbol\beta_t-\hat{\boldsymbol\beta})$ 는 $\rho^{2t}$ 로 줄어요.)

→ **조건수 $\kappa$ 가 크면 느리게 수렴**해요. 특성을 **표준화**하면 절편 열과 직교하고 스케일이 같아져요. 특성이 하나뿐인 이 예제에서는 $A=\operatorname{diag}\big(2,\ 2\tfrac{n-1}{n}\big)$ 이라서 $\kappa=\tfrac{n}{n-1}\approx1$ 이에요.
특성이 여러 개면 $\kappa$ 는 대략 **특성 상관행렬의 조건수**가 돼요. 특성끼리 상관이 강하면 표준화해도 $\kappa$ 가 커요 (PART 3에서 확인해요).

In [ ]:
def mse_loss(beta, X, y):
    return ((y - X @ beta) ** 2).mean()

x_mu, x_sd = x.mean(), x.std()
Xs = torch.stack([torch.ones(n, dtype=F64), (x - x_mu) / x_sd], dim=1)   # 표준화한 설계행렬

lam_max = {}
for name, M_ in [("raw", X), ("standardized", Xs)]:
    lam = torch.linalg.eigvalsh(2 / n * M_.T @ M_)
    lam_max[name] = lam[-1].item()
    print(f"{name:13s} λ = {lam.numpy().round(4)}   κ = {(lam[-1] / lam[0]).item():7.1f}   2/λmax = {2 / lam[-1].item():.4f}")

In [ ]:
def gradient_descent(X, y, lr, steps):
    """autograd로 기울기를 구하는 가장 기본적인 경사하강법"""
    beta = torch.zeros(X.shape[1], dtype=F64, requires_grad=True)
    path, losses = [beta.detach().clone()], []
    for _ in range(steps):
        loss = mse_loss(beta, X, y)
        losses.append(loss.item())
        loss.backward()                      # beta.grad ← ∇J(β)
        with torch.no_grad():
            beta -= lr * beta.grad
            beta.grad.zero_()
        path.append(beta.detach().clone())
    return torch.stack(path), torch.tensor(losses, dtype=F64)

J_star = mse_loss(beta_ls, X, y).item()
runs = {
    r"raw, $\eta = 1.0/\lambda_{max}$":           (X,  1.00 / lam_max["raw"]),
    r"raw, $\eta = 1.9/\lambda_{max}$":           (X,  1.90 / lam_max["raw"]),
    r"raw, $\eta = 2.02/\lambda_{max}$ (diverge)": (X,  2.02 / lam_max["raw"]),
    r"standardized, $\eta = 1/\lambda_{max}$":     (Xs, 1.00 / lam_max["standardized"]),
}
results = {k: gradient_descent(M_, y, lr, steps=300) for k, (M_, lr) in runs.items()}

# 원래 좌표 (β0, β1) 에서의 손실 등고선
b0g, b1g = torch.meshgrid(torch.linspace(-0.5, 3.0, 160, dtype=F64),
                          torch.linspace(-0.05, 0.9, 160, dtype=F64), indexing="ij")
Jg = ((y - b0g[..., None] - b1g[..., None] * x) ** 2).mean(-1)

def to_original(path_std):
    """표준화 좌표의 (b, w) 경로 → 원래 좌표 (β0, β1)"""
    b_, w_ = path_std[:, 0], path_std[:, 1]
    return torch.stack([b_ - w_ * x_mu / x_sd, w_ / x_sd], dim=1)

keys = list(results)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
ax = axes[0]
ax.contour(b0g.numpy(), b1g.numpy(), Jg.numpy(), levels=(J_star + torch.logspace(-2.5, 1.2, 16)).numpy(),
           cmap="Greys_r", linewidths=0.6)
for k, c, p in [(keys[0], "C0", results[keys[0]][0]), (keys[1], "C1", results[keys[1]][0]),
                (keys[3], "C2", to_original(results[keys[3]][0]))]:
    ax.plot(p[:, 0].numpy(), p[:, 1].numpy(), ".-", ms=2.5, lw=0.7, color=c, label=k + " (300 steps)")
ax.plot(*beta_ls.numpy(), "k*", ms=12, label=r"$\hat\beta$")
ax.set(xlim=(-0.5, 3.0), ylim=(-0.05, 0.9), xlabel=r"$\beta_0$", ylabel=r"$\beta_1$",
       title=r"GD paths in the original $(\beta_0, \beta_1)$ space")
ax.legend(fontsize=7, loc="upper right")

ax = axes[1]
for (k, (_, losses)), c in zip(results.items(), ["C0", "C1", "C3", "C2"]):
    ax.semilogy((losses - J_star).clamp_min(1e-16).numpy(), color=c, label=k)
ax.set(xlabel="iteration t", ylabel=r"$J(\beta_t) - J^\star$", ylim=(1e-12, 1e4), title="Convergence (log scale)")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

- 파란색($\eta=1/\lambda_{\max}$)과 주황색($\eta=1.9/\lambda_{\max}$) 모두 긴 골짜기(작은 고윳값 방향)를 따라 느리게 움직여요. 수렴률 $\max_i|1-\eta\lambda_i|$ 가 각각 $1-1/\kappa\approx0.992$, $1-1.9/\kappa\approx0.984$ 라서, **지그재그가 심한 주황색이 오히려 더 빨라요**.
- $\eta=2.02/\lambda_{\max}$ 는 $|1-\eta\lambda_{\max}|>1$ 이라 발산해요.
- 표준화하면 $\kappa\approx1$ 이라 몇 걸음 만에 수렴해요 (초록색).

**PyTorch 표준 학습 루프**: `nn.Linear` + `DataLoader`(미니배치) + `torch.optim.SGD`.
미니배치 기울기는 전체 기울기의 **불편추정량**이에요. 학습률이 일정하면 최적점 근처에서 조금씩 흔들려요.
표준화된 공간에서 학습한 $(w, b)$ 는 $\beta_1=w/s_x$, $\beta_0=b-w\,\bar x/s_x$ 로 원래 스케일로 되돌려요.

PART 3에서 같은 구조의 학습 루프를 실제 데이터에 써요. (60분 진행이면 이 셀은 건너뛰고 PART 3에서 설명해요.)

In [ ]:
torch.manual_seed(0)
ds = TensorDataset(((x - x_mu) / x_sd).float().unsqueeze(1), y.float().unsqueeze(1))
loader = DataLoader(ds, batch_size=20, shuffle=True, generator=torch.Generator().manual_seed(0))

model = nn.Linear(1, 1)                       # ŷ = w·x + b
opt = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()

epoch_loss = []
for epoch in range(60):
    total = 0.0
    for xb, yb in loader:
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    epoch_loss.append(total / len(ds))

w, b = model.weight.item(), model.bias.item()
beta1_sgd = w / x_sd.item()
beta0_sgd = b - w * x_mu.item() / x_sd.item()
print(f"mini-batch SGD : β0 = {beta0_sgd:.4f}, β1 = {beta1_sgd:.4f}")
print(f"closed form    : β0 = {beta_ls[0].item():.4f}, β1 = {beta_ls[1].item():.4f}")

fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(epoch_loss, label="train MSE (epoch mean)")
ax.axhline(J_star, color="k", ls="--", label=r"$J^\star$ (closed form)")
ax.set(xlabel="epoch", ylabel="MSE", yscale="log", title="nn.Linear + SGD")
ax.legend()
plt.show()

### 1-7. 기저함수 확장 · 과적합 · 릿지
특성 사상 $\boldsymbol\phi:\mathbb R\to\mathbb R^q$, 예를 들어 $\boldsymbol\phi(x)=(x,x^2,\dots,x^d)$ 를 써도 $y=\beta_0+\boldsymbol\phi(x)^\top\mathbf w+\varepsilon$ 은 **$\mathbf w$ 에 대해 선형**이에요. 그래서 지금까지의 이론이 그대로 적용돼요.

차수 $d$ 를 키우면 학습 오차는 절대 늘지 않아요(모형이 중첩되니까요). 하지만 새 데이터에 대한 오차는 **편향–분산 분해**를 따라요.

$$\mathbb E\big[(y_0-\hat f(x_0))^2\big]=\underbrace{\sigma^2}_{\text{줄일 수 없음}}+\underbrace{\big(f(x_0)-\mathbb E\hat f(x_0)\big)^2}_{\text{bias}^2}+\underbrace{\operatorname{Var}\hat f(x_0)}_{\text{variance}}$$

**릿지 회귀**는 계수 크기에 벌점을 줘서 분산을 줄여요. 특성은 표준화한 $Z$ 를 쓰고, 절편은 벌점 없이 $\bar y$ 로 둬요 ($\mathbf y_c=\mathbf y-\bar y\mathbf 1$).

$$\min_{\mathbf w}\ \frac1n\lVert\mathbf y_c-Z\mathbf w\rVert^2+\lambda\lVert\mathbf w\rVert^2
\quad\Longrightarrow\quad
\hat{\mathbf w}_\lambda=(Z^\top Z+n\lambda I)^{-1}Z^\top\mathbf y_c$$

$Z^\top Z+n\lambda I\succ0$ 이라서 $q>n$ 이어도 해가 유일해요. SVD $Z=UDV^\top$ 로 쓰면

$$Z\hat{\mathbf w}_\lambda=\sum_j\mathbf u_j\,\frac{d_j^2}{d_j^2+n\lambda}\,\mathbf u_j^\top\mathbf y_c,
\qquad \operatorname{df}(\lambda)=\sum_j\frac{d_j^2}{d_j^2+n\lambda}$$

즉 **작은 특이값 방향을 강하게 줄여요(shrinkage)**. 유효 자유도 $\operatorname{df}(\lambda)$ 는 $\lambda\to0$ 일 때 $\operatorname{rank}(Z)$ 에서 $\lambda\to\infty$ 일 때 $0$ 까지 연속적으로 줄어요. $Z$ 가 열 완전계수면 $\operatorname{rank}(Z)=q$ 예요. 다만 $Z$ 는 열 평균을 뺀 행렬이라 $\operatorname{rank}(Z)\le n-1$ 이므로, $q\ge n$ 이면 $q$ 보다 작아요.

아래 `ridge_fit` · `ridge_predict`는 PART 4에서도 그대로 써요.

In [ ]:
def poly_1d(x, d):
    """[x, x², …, x^d]  (상수항 제외 — 절편은 따로 다뤄요)"""
    if d == 0:
        return torch.zeros(len(x), 0, dtype=x.dtype)
    return torch.stack([x ** k for k in range(1, d + 1)], dim=1)

def ridge_fit(P, y, lam=0.0):
    """특성 P(상수항 제외)를 표준화하고 릿지 회귀를 풀어요. 절편은 벌점 없음, lam=0 이면 OLS."""
    b = y.mean()
    if P.shape[1] == 0:                                    # 절편만 있는 모형
        return {"mu": P.sum(0), "sd": P.sum(0) + 1, "w": P.sum(0), "b": b}
    mu, sd = P.mean(0), P.std(0).clamp_min(1e-12)
    Z = (P - mu) / sd
    if lam == 0:
        w = torch.linalg.lstsq(Z, (y - b).unsqueeze(1)).solution.squeeze(1)
    else:
        A = Z.T @ Z + lam * len(y) * torch.eye(P.shape[1], dtype=P.dtype)
        w = torch.linalg.solve(A, Z.T @ (y - b))
    return {"mu": mu, "sd": sd, "w": w, "b": b}

def ridge_predict(m, P):
    return m["b"] + ((P - m["mu"]) / m["sd"]) @ m["w"]

def mse(a, b):
    return ((a - b) ** 2).mean()

# 참 함수 f(x) = sin(πx), 학습 25개 · 테스트 2,000개, 잡음 σ = 0.3
g = torch.Generator().manual_seed(3)
f_true = lambda t: torch.sin(torch.pi * t)
x_tr = 2 * torch.rand(25, generator=g, dtype=F64) - 1
y_tr = f_true(x_tr) + 0.3 * torch.randn(25, generator=g, dtype=F64)
x_te = 2 * torch.rand(2000, generator=g, dtype=F64) - 1
y_te = f_true(x_te) + 0.3 * torch.randn(2000, generator=g, dtype=F64)

degrees = list(range(16))
fits_1d = {d: ridge_fit(poly_1d(x_tr, d), y_tr) for d in degrees}
tr_err = [mse(ridge_predict(fits_1d[d], poly_1d(x_tr, d)), y_tr).item() for d in degrees]
te_err = [mse(ridge_predict(fits_1d[d], poly_1d(x_te, d)), y_te).item() for d in degrees]

xg = torch.linspace(-1, 1, 400, dtype=F64)
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].semilogy(degrees, tr_err, "o-", label="train MSE")
axes[0].semilogy(degrees, te_err, "s-", label="test MSE")
axes[0].axhline(0.3**2, color="k", ls=":", label=r"noise $\sigma^2$")
axes[0].set(xlabel="polynomial degree d", ylabel="MSE", title="Bias–variance trade-off")
axes[0].legend()
axes[1].scatter(x_tr.numpy(), y_tr.numpy(), color="k", s=15, zorder=3, label="train data")
axes[1].plot(xg.numpy(), f_true(xg).numpy(), "k--", lw=1, label="true f")
for d, c in [(1, "C0"), (3, "C2"), (15, "C3")]:
    axes[1].plot(xg.numpy(), ridge_predict(fits_1d[d], poly_1d(xg, d)).numpy(), color=c, label=f"d = {d}")
axes[1].set(ylim=(-2, 2), xlabel="x", ylabel="y", title="Under- vs over-fitting")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

같은 $d=15$ 모형에 릿지 벌점을 주면 차수를 낮추지 않고도 과적합을 막을 수 있어요. (60분 진행이면 이 셀은 건너뛰고 PART 4-4에서 다뤄요.)

In [ ]:
d_big = 15
P_tr, P_te, P_g = poly_1d(x_tr, d_big), poly_1d(x_te, d_big), poly_1d(xg, d_big)
lams = torch.logspace(-8, 1, 19, dtype=F64)
te_ridge = torch.tensor([mse(ridge_predict(ridge_fit(P_tr, y_tr, l.item()), P_te), y_te).item() for l in lams])
lam_best = lams[te_ridge.argmin()].item()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].loglog(lams.numpy(), te_ridge.numpy(), "o-")
axes[0].axhline(0.3**2, color="k", ls=":")
axes[0].set(xlabel=r"$\lambda$", ylabel="test MSE", title=f"Ridge (d = {d_big}): best λ ≈ {lam_best:.1e}")
axes[1].scatter(x_tr.numpy(), y_tr.numpy(), color="k", s=15, zorder=3)
axes[1].plot(xg.numpy(), f_true(xg).numpy(), "k--", lw=1, label="true f")
for lam, c in [(1e-8, "C3"), (lam_best, "C2"), (10.0, "C0")]:
    axes[1].plot(xg.numpy(), ridge_predict(ridge_fit(P_tr, y_tr, lam), P_g).numpy(), color=c, label=f"λ = {lam:.0e}")
axes[1].set(ylim=(-2, 2), xlabel="x", title=f"Ridge fits, d = {d_big}")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

> 여기서는 이해를 돕기 위해 **테스트 오차로 직접** $d$, $\lambda$ 를 봤어요. 실제로는 테스트 데이터를 건드리지 않고 **교차검증(cross-validation)** 으로 골라요 → PART 4.

---
# __PART 2. WM-811K 데이터 탐색__

### 2-1. 데이터셋 소개
**WM-811K**는 실제 반도체 제조 공정에서 수집한 **웨이퍼 빈 맵(wafer bin map)** 데이터셋이에요 (Wu, Jang & Chen, *IEEE Trans. Semiconductor Manufacturing*, 2015).

- 웨이퍼 **811,457장**, 로트(lot) **46,293개** (로트 = 보통 25장 단위로 함께 공정을 거치는 웨이퍼 묶음)
- 각 웨이퍼 맵은 2차원 정수 배열: `0` = 웨이퍼 밖, `1` = 정상 다이(die), `2` = 불량 다이
- 전문가가 172,950장에 9가지 **불량 패턴** 라벨을 달았어요 (`none` 포함). 나머지 약 79%는 라벨이 없어요.

| 열 | 내용 |
|---|---|
| `waferMap` | 웨이퍼 맵 (크기가 제품마다 다름) |
| `dieSize` | 웨이퍼 안 다이 수 |
| `lotName` | 로트 이름 |
| `waferIndex` | 로트 안에서의 웨이퍼 번호 (1–25) |
| `trianTestLabel` | 원 논문의 학습/테스트 구분 (열 이름의 오타는 원본 그대로) |
| `failureType` | 불량 패턴 라벨 |

원래는 **분류** 문제용 데이터지만, 이번 시간에는 **수율(yield)** 과 **불량 확률**이라는 연속형 타깃을 만들어서 **회귀** 문제로 다뤄요.
$$\text{yield}=\frac{\#\{\text{정상 다이}\}}{\#\{\text{전체 다이}\}}=1-\frac{\#\{\text{불량 다이}\}}{\#\{\text{전체 다이}\}}$$

### 2-2. 안전하게 불러오기
`LSWMD.pkl`은 **Python 2 + pandas 0.x 시절의 pickle**이에요. 그래서 그냥 `pd.read_pickle`로 열면
`ModuleNotFoundError: No module named 'pandas.indexes'` 오류가 나요 (모듈 경로가 바뀌었기 때문이에요).

또 pickle은 **열 때 임의의 코드를 실행할 수 있는** 형식이라, 인터넷에서 받은 pickle을 그대로 여는 건 위험해요.
그래서 이 파일이 실제로 참조하는 **10개 클래스만 허용**하는 `Unpickler`를 만들고, 바뀐 모듈 경로도 여기서 연결해 줘요.

In [ ]:
import pandas.core.indexes.base as _ibase
import pandas.core.internals as _internals
try:                                       # numpy ≥ 2.0
    from numpy._core import multiarray as _multiarray
except ImportError:                        # numpy 1.x
    from numpy.core import multiarray as _multiarray


class WaferUnpickler(pickle.Unpickler):
    """LSWMD.pkl 전용: 파일이 참조하는 클래스만 허용하고, 옛 모듈 경로를 새 경로로 연결해요."""
    ALLOWED = {
        ("pandas.core.frame", "DataFrame"): pd.DataFrame,
        ("pandas.core.internals", "BlockManager"): _internals.BlockManager,
        ("pandas.indexes.base", "_new_Index"): _ibase._new_Index,       # pandas.indexes → pandas.core.indexes
        ("pandas.indexes.base", "Index"): pd.Index,
        ("pandas.indexes.range", "RangeIndex"): pd.RangeIndex,
        ("numpy", "ndarray"): np.ndarray,
        ("numpy", "dtype"): np.dtype,
        ("numpy.core.multiarray", "_reconstruct"): _multiarray._reconstruct,
        ("numpy.core.multiarray", "scalar"): _multiarray.scalar,
        ("__builtin__", "slice"): slice,                                # Python 2 → 3
    }

    def find_class(self, module, name):
        if (module, name) not in self.ALLOWED:
            raise pickle.UnpicklingError(f"허용되지 않은 객체: {module}.{name}")
        return self.ALLOWED[(module, name)]


def load_wm811k(path):
    with open(path, "rb") as f:
        return WaferUnpickler(f, encoding="latin1").load()   # Python 2 문자열 → latin1


t0 = time.time()
raw = load_wm811k(PKL_PATH)
print(f"불러오기 {time.time() - t0:.1f}초, shape = {raw.shape}")
raw.head(3)

### 2-3. 정리하기
`failureType`, `trianTestLabel` 은 `[['none']]` 처럼 2차원 배열로 감싸져 있고, 라벨이 없으면 빈 배열이에요. 값을 꺼내서 깔끔한 데이터프레임 `df` 를 만들어요.

In [ ]:
def unwrap(v):
    """[['none']] → 'none',  빈 배열(라벨 없음) → None"""
    return v.item() if getattr(v, "size", 0) == 1 else None

df = pd.DataFrame({
    "waferMap":    raw["waferMap"],
    "lot":         raw["lotName"],
    "wafer_index": raw["waferIndex"].astype(int),
    "split":       raw["trianTestLabel"].map(unwrap).fillna("unlabeled"),
    "label":       raw["failureType"].map(unwrap).fillna("unlabeled"),
    "die_size":    raw["dieSize"].astype(int),
})
df["H"] = df["waferMap"].map(lambda a: a.shape[0])
df["W"] = df["waferMap"].map(lambda a: a.shape[1])

CLASSES = ["none", "Center", "Donut", "Edge-Loc", "Edge-Ring", "Loc", "Near-full", "Random", "Scratch"]
print(f"웨이퍼 {len(df):,}장 · 로트 {df['lot'].nunique():,}개 · 맵 크기(제품) {df.groupby(['H', 'W']).ngroups}종")
print(f"맵에 들어 있는 값: {sorted(set(df['waferMap'].iloc[0].ravel().tolist()))}")
display(pd.crosstab(df["label"], df["split"]).loc[CLASSES + ["unlabeled"]])

In [ ]:
counts = df["label"].value_counts().reindex(CLASSES)
fig, ax = plt.subplots(figsize=(9, 3.2))
bars = ax.bar(counts.index, counts.to_numpy(), color=["C7"] + ["C0"] * 8)
ax.bar_label(bars, labels=[f"{v:,}" for v in counts.to_numpy()], fontsize=8)
ax.set(yscale="log", ylabel="# wafers (log)", title="Labeled wafers per failure type (unlabeled: {:,})".format((df['label'] == 'unlabeled').sum()))
plt.xticks(rotation=30)
plt.show()

> 라벨 분포가 **매우 불균형**해요 (`none` 147,431장 vs `Near-full` 149장). 분류 문제를 다룰 때 중요한 이슈예요 (다음 시간들).

### 2-4. 웨이퍼 맵 시각화

In [ ]:
WAFER_CMAP = ListedColormap(["#ffffff", "#c6dbef", "#cb181d"])   # 0: 웨이퍼 밖, 1: 정상, 2: 불량

def show_wafer(ax, wm, title=None):
    """웨이퍼 맵을 정규화 좌표 [-1, 1]² 위에 그려요 (다이가 직사각형이어도 웨이퍼가 원으로 보여요)"""
    ax.imshow(wm, cmap=WAFER_CMAP, vmin=0, vmax=2, interpolation="nearest", extent=(-1, 1, -1, 1))
    ax.set_aspect("equal")
    ax.set_axis_off()
    if title:
        ax.set_title(title, fontsize=8)

g = torch.Generator().manual_seed(1)
n_show = 4
fig, axes = plt.subplots(n_show, len(CLASSES), figsize=(1.7 * len(CLASSES), 1.8 * n_show))
for j, c in enumerate(CLASSES):
    idx = torch.tensor(df.index[df["label"] == c].to_numpy())
    for i, k in enumerate(idx[torch.randperm(len(idx), generator=g)[:n_show]].tolist()):
        wm = df.at[k, "waferMap"]
        wm_t = torch.from_numpy(wm)
        y_k = 1 - (wm_t == 2).sum().item() / (wm_t > 0).sum().item()
        show_wafer(axes[i, j], wm, f"{c}\n{wm.shape[0]}×{wm.shape[1]}, yield {y_k:.2f}" if i == 0 else f"yield {y_k:.2f}")
plt.suptitle("WM-811K wafer maps (white: outside, blue: good die, red: defective die)", y=1.0)
plt.tight_layout()
plt.show()

### 2-5. 웨이퍼 단위 특징 계산 (PyTorch)
맵 크기가 제품마다 달라서, **같은 크기 $(H, W)$ 끼리 묶어 3차원 텐서로 쌓은 뒤** 한 번에 계산해요.

**정규화 좌표**: 격자 $(i, j)$ 를 웨이퍼 중심이 원점, 반지름이 1이 되도록 옮겨요. 다이가 직사각형이라도 웨이퍼는 단위원이 돼요.
$$x=\frac{j-(W-1)/2}{W/2},\qquad y=\frac{(H-1)/2-i}{H/2},\qquad r=\sqrt{x^2+y^2}$$

웨이퍼마다 다음을 구해요.
- `n_die`, `n_def`: 전체 · 불량 다이 수 → `yield`
- `die_r{k}`, `def_r{k}`: 반지름 구간 $[k/10,(k+1)/10)$ 의 다이 · 불량 수 ($k=0,\dots,9$, 경계의 $r\ge1$ 인 0.6%는 마지막 구간에 넣어요)

각 구간 소속을 one-hot 텐서 $(H,W,10)$ 로 만들면, 구간별 개수는 `einsum` 한 번으로 구할 수 있어요.

In [ ]:
N_BINS = 10
R_EDGES = torch.linspace(0, 1, N_BINS + 1, dtype=F64)
FEAT_COLS = ["n_die", "n_def"] + [f"die_r{k}" for k in range(N_BINS)] + [f"def_r{k}" for k in range(N_BINS)]
MAPS = df["waferMap"].to_numpy()          # 웨이퍼 맵 배열들 (object array)

def normalized_grid(H, W):
    """맵 격자의 정규화 좌표 (x, y): 중심 0, 반지름 1"""
    ii, jj = torch.meshgrid(torch.arange(H, dtype=F64), torch.arange(W, dtype=F64), indexing="ij")
    return (jj - (W - 1) / 2) / (W / 2), ((H - 1) / 2 - ii) / (H / 2)

def stack_maps(idx):
    """같은 크기 맵들을 (m, H, W) uint8 텐서로"""
    return torch.stack([torch.from_numpy(MAPS[k]) for k in idx])

feats = torch.zeros(len(df), len(FEAT_COLS), dtype=F64)
t0 = time.time()
for (H, W), idx in df.groupby(["H", "W"]).indices.items():
    xg_, yg_ = normalized_grid(H, W)
    rbin = torch.bucketize(torch.sqrt(xg_**2 + yg_**2).clamp(max=1 - 1e-9), R_EDGES[1:-1])  # 0 … 9
    onehot = nn.functional.one_hot(rbin, N_BINS).float()                                     # (H, W, 10)
    for s in range(0, len(idx), 20_000):                       # 메모리를 아끼려고 나눠서 처리
        part = torch.from_numpy(idx[s:s + 20_000])
        maps = stack_maps(part.tolist())
        die, bad = (maps > 0).float(), (maps == 2).float()
        feats[part, 0] = die.sum((1, 2)).double()
        feats[part, 1] = bad.sum((1, 2)).double()
        feats[part, 2:2 + N_BINS] = torch.einsum("mhw,hwk->mk", die, onehot).double()
        feats[part, 2 + N_BINS:] = torch.einsum("mhw,hwk->mk", bad, onehot).double()
print(f"특징 계산 {time.time() - t0:.1f}초")

wafer = pd.concat([df[["lot", "wafer_index", "split", "label", "H", "W"]],
                   pd.DataFrame(feats.numpy(), columns=FEAT_COLS, index=df.index)], axis=1)
wafer["yield"] = (1 - feats[:, 1] / feats[:, 0]).numpy()          # 수율 = 1 − 불량 수 / 다이 수
assert (wafer["n_die"].to_numpy() == df["die_size"].to_numpy()).all()   # dieSize = 웨이퍼 안 다이 수 ✔
wafer.head()

### 2-6. 수율 분포

In [ ]:
yield_all = torch.tensor(wafer["yield"].to_numpy(), dtype=F64)
print(f"평균 수율 {yield_all.mean().item():.4f}, 중앙값 {yield_all.median().item():.4f}, 표준편차 {yield_all.std().item():.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1, 1.4]})
axes[0].hist(yield_all.numpy(), bins=100, color="C0")
axes[0].set(xlabel="yield", ylabel="# wafers", yscale="log", title="Yield distribution (all 811,457 wafers)")
data = [torch.tensor(wafer.loc[wafer["label"] == c, "yield"].to_numpy()) for c in CLASSES]
axes[1].boxplot([d.numpy() for d in data], showfliers=False)
axes[1].set_xticks(range(1, len(CLASSES) + 1), CLASSES, rotation=30)
axes[1].set(ylabel="yield", title="Yield by failure type (labeled wafers)")
plt.tight_layout()
plt.show()

wafer[wafer["label"] != "unlabeled"].groupby("label")["yield"].agg(["size", "mean", "median", "std"]).loc[CLASSES]

- 수율은 1 쪽에 몰려 있고 왼쪽으로 긴 꼬리가 있어요 (평균 0.903 < 중앙값 0.929). 세로축이 로그라서 꼬리가 크게 보인다는 점에 주의하세요.
- 0.18 근처의 뾰족한 막대(약 1,200장)는 거의 **한 제품(맵 122×102)** 의 웨이퍼예요. 수율이 정확히 0인 웨이퍼도 339장 있어요. 이런 덩어리는 2-7에서 이상치로 다시 나타나요.
- 라벨이 `none` 인 웨이퍼의 평균 수율(0.894)이 라벨 없는 웨이퍼(0.909)보다 낮아요. 라벨을 붙인 웨이퍼가 무작위 표본이 아닐 수 있다는 뜻이에요.

### 2-7. 데이터 분석 사례: 다이 크기와 수율 (Poisson 수율 모형)
반도체 공정의 고전적인 수율 모형은 결함이 웨이퍼 위에 밀도 $D_0$ 인 **포아송 과정**으로 떨어진다고 가정해요. 면적 $A$ 인 다이에 결함이 하나도 없을 확률은

$$Y=P(\text{결함 }0\text{개})=e^{-AD_0}.$$

웨이퍼 면적 $A_w$ 가 같다면 다이 면적은 $A\approx A_w/N$ ($N$ = 웨이퍼당 다이 수)이에요. 공정 전체에 걸친 수율 손실 $Y_0$ 까지 넣으면

$$Y=Y_0\,e^{-A_wD_0/N}\quad\Longrightarrow\quad \ln Y=\underbrace{\ln Y_0}_{\beta_0}+\underbrace{(-A_wD_0)}_{\beta_1}\cdot\frac1N$$

**비선형 모형을 변환해서 선형회귀로** 바꿨어요. 웨이퍼가 300장 이상인 제품(맵 크기)별 평균 수율로 확인해 봐요.

In [ ]:
prod = (wafer.groupby(["H", "W"])
        .agg(n_wafers=("yield", "size"), N=("n_die", "median"), Y=("yield", "mean"))
        .query("n_wafers >= 300").reset_index())
u = 1 / torch.tensor(prod["N"].to_numpy(), dtype=F64)                 # 특성: 1/N
v = torch.log(torch.tensor(prod["Y"].to_numpy(), dtype=F64))          # 타깃: ln Y
fit_p = ols(torch.stack([torch.ones_like(u), u], 1), v)
b0, b1 = fit_p["beta"].tolist()
print(f"제품 {len(prod)}개")
print(f"ln Y = {b0:.4f} + ({b1:.1f})·(1/N)   →  Y0 = {math.exp(b0):.3f},  A_w·D0 = {-b1:.1f}")
print(f"slope t = {fit_p['t'][1].item():.2f},  R² = {fit_p['r2'].item():.3f}")

Ng = torch.logspace(math.log10(prod["N"].min()), math.log10(prod["N"].max()), 200, dtype=F64)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
sc = ax.scatter(prod["N"], prod["Y"], s=8 + 40 * prod["n_wafers"] / prod["n_wafers"].max(), alpha=0.6, label="product (map size)")
ax.plot(Ng.numpy(), (math.exp(b0) * torch.exp(b1 / Ng)).numpy(), "r-", label=r"fit: $Y_0\,e^{-A_wD_0/N}$")
ax.set(xscale="log", xlabel="dies per wafer N (log)", ylabel="mean yield", title=f"Poisson yield model across products (R² on ln Y = {fit_p['r2'].item():.3f})")
ax.legend()
plt.show()

# 영향점 확인: 평균 수율이 0.5 미만인 제품을 빼고 다시 적합
keep = torch.tensor((prod["Y"] >= 0.5).to_numpy())
resid_p = v - torch.stack([torch.ones_like(u), u], 1) @ fit_p["beta"]
fit_k = ols(torch.stack([torch.ones_like(u), u], 1)[keep], v[keep])
print(f"평균 수율 0.5 미만 제품 {(~keep).sum().item()}개가 잔차제곱합에서 차지하는 비율: "
      f"{((resid_p[~keep] ** 2).sum() / (resid_p ** 2).sum()).item():.0%}")
print(f"이 {(~keep).sum().item()}개를 빼면: slope = {fit_k['beta'][1].item():.1f},  t = {fit_k['t'][1].item():.2f},  R² = {fit_k['r2'].item():.3f}")

- 기울기의 부호(다이가 클수록 = $N$ 이 작을수록 수율이 낮음)는 이론과 맞고, $t$ 값으로 보면 통계적으로도 유의해요.
- 하지만 $R^2$ 가 매우 작아요. 그림 아래쪽의 **평균 수율이 0.5도 안 되는 제품 5개**가 $\ln Y$ 척도에서 직선과 아주 멀리 떨어져서($\ln 0.19\approx-1.7$) 잔차제곱합의 대부분을 차지하기 때문이에요. 이 5개를 빼면 기울기는 거의 그대로인데 $R^2$ 와 $|t|$ 가 크게 올라가요 (**영향점**: 로그 변환은 작은 값을 크게 벌려 놓아요).
- 5개를 빼도 변동의 60% 이상은 설명되지 않아요. 제품마다 공정 · 설계 · 성숙도가 달라서 $D_0$ 자체가 다르기 때문이라고 볼 수 있어요 (**누락 변수**). 아래쪽 5개는 그 극단적인 경우예요. (여기서 '제품'은 맵 크기로 대신한 거예요.)
- **"이론적으로 타당한 모형" ≠ "설명력이 높은 모형"**. 회귀분석에서는 데이터가 모형을 얼마나 지지하는지, 결과가 몇 개의 점에 좌우되지 않는지 항상 확인해야 해요.

> 💬 $Y_0<1$ 은 물리적으로 무엇을 뜻할까요? (힌트: 면적과 무관한 수율 손실)

### 2-8. 로트 효과: 같은 로트의 웨이퍼는 닮았다
일원분산분석(one-way ANOVA)으로 수율 변동을 **로트 간 / 로트 내**로 나눠요.

$$\underbrace{\sum_{g,i}(y_{gi}-\bar y)^2}_{\text{SST}}=\underbrace{\sum_g n_g(\bar y_g-\bar y)^2}_{\text{SSB (로트 간)}}+\underbrace{\sum_{g,i}(y_{gi}-\bar y_g)^2}_{\text{SSW (로트 내)}},\qquad \eta^2=\frac{\text{SSB}}{\text{SST}}$$

이건 **로트 더미(one-hot) 변수 $G$ 개로 회귀한 것의 $R^2$** 와 같아요. 로트 지시 벡터들이 만드는 열공간으로 사영하면 각 로트의 평균이 되니까요 (1-4의 사영).
로트가 46,293개나 되니까 자유도를 보정한 $R^2_{adj}=1-\dfrac{\text{SSW}/(n-G)}{\text{SST}/(n-1)}$ 도 함께 봐요.

In [ ]:
lot_id = torch.from_numpy(pd.factorize(wafer["lot"])[0])
G = int(lot_id.max()) + 1
n_w = len(yield_all)
cnt = torch.zeros(G, dtype=F64).index_add_(0, lot_id, torch.ones_like(yield_all))
lot_mean = torch.zeros(G, dtype=F64).index_add_(0, lot_id, yield_all) / cnt     # 로트별 평균 (scatter-add)

sst = ((yield_all - yield_all.mean()) ** 2).sum()
ssw = ((yield_all - lot_mean[lot_id]) ** 2).sum()
print(f"로트 {G:,}개:  η² = 1 − SSW/SST = {(1 - ssw / sst).item():.3f},   R²_adj = {(1 - (ssw / (n_w - G)) / (sst / (n_w - 1))).item():.3f}")

# 비교: 로트가 아니라 제품(맵 크기)으로만 묶으면?
prod_id = torch.tensor(wafer.groupby(["H", "W"]).ngroup().to_numpy())
Pn = int(prod_id.max()) + 1
cnt_p = torch.zeros(Pn, dtype=F64).index_add_(0, prod_id, torch.ones_like(yield_all))
prod_mean = torch.zeros(Pn, dtype=F64).index_add_(0, prod_id, yield_all) / cnt_p
ssw_p = ((yield_all - prod_mean[prod_id]) ** 2).sum()
print(f"제품 {Pn}종:  η² = {(1 - ssw_p / sst).item():.3f},   같은 제품 안의 변동 중 로트가 설명하는 비율 = {((ssw_p - ssw) / ssw_p).item():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw={"width_ratios": [1.5, 1]})
g = torch.Generator().manual_seed(2)
lots_all = wafer["lot"].unique()
pick = lots_all[torch.randperm(len(lots_all), generator=g)[:40].numpy()]
sub = wafer[wafer["lot"].isin(pick)]
order = sub.groupby("lot")["yield"].mean().sort_values().index
for i, l in enumerate(order):
    yl = sub.loc[sub["lot"] == l, "yield"].to_numpy()
    axes[0].scatter([i] * len(yl), yl, s=6, alpha=0.6, color=f"C{i % 10}")
axes[0].set(xlabel="40 random lots (sorted by lot mean)", ylabel="wafer yield", title="Wafers cluster by lot")
axes[0].set_xticks([])

widx = torch.tensor(wafer["wafer_index"].to_numpy()) - 1                       # 0 … 24
cnt_i = torch.zeros(25, dtype=F64).index_add_(0, widx, torch.ones_like(yield_all))
mean_i = torch.zeros(25, dtype=F64).index_add_(0, widx, yield_all) / cnt_i
var_i = torch.zeros(25, dtype=F64).index_add_(0, widx, (yield_all - mean_i[widx]) ** 2) / (cnt_i - 1)
se_i = torch.sqrt(var_i / cnt_i)
axes[1].errorbar(range(1, 26), mean_i.numpy(), yerr=(2 * se_i).numpy(), fmt="o-", ms=3, capsize=2)
axes[1].set(xlabel="wafer index in lot", ylabel="mean yield (±2 se)", title="Position in lot barely matters")
plt.tight_layout()
plt.show()

- 수율 변동의 대부분(η²≈0.91)이 **로트 간 차이**에서 와요. 다만 로트마다 제품(맵 크기)이 정해져 있어서, 제품만으로도 η²≈0.53이 설명돼요. 같은 제품 안에서 남은 변동도 대부분 로트 간 차이예요. 같은 로트의 웨이퍼는 같은 장비 · 조건에서 함께 공정을 거치니까 자연스러운 결과예요.
- 반면 로트 안에서의 **위치(웨이퍼 번호)** 는 평균 수율과 거의 관계가 없어요. 오른쪽 그림의 **y축 범위가 0.898–0.905로 아주 좁다**는 점에 주의하세요. 양 끝(1–2번, 24–25번) 웨이퍼가 조금 낮지만 차이는 0.5%p 이내예요.
- → PART 3: **"로트의 앞쪽 몇 장만 검사하면 나머지 웨이퍼의 수율을 예측할 수 있을까?"**

### 2-9. 반지름에 따른 불량률 (불량 패턴별)
패턴별로 반지름 구간의 불량률 $\sum\text{def}_k / \sum\text{die}_k$ 를 그려 봐요.

In [ ]:
die_cols = [f"die_r{k}" for k in range(N_BINS)]
def_cols = [f"def_r{k}" for k in range(N_BINS)]
grp_def = wafer.groupby("label")[def_cols].sum()            # 패턴별 · 구간별 불량 수 (집계)
grp_die = wafer.groupby("label")[die_cols].sum()            # 패턴별 · 구간별 다이 수
rate = torch.tensor(grp_def.to_numpy()) / torch.tensor(grp_die.to_numpy())
radial_rate = pd.DataFrame(rate.numpy(), index=grp_def.index, columns=[f"{k/10:.1f}–{(k+1)/10:.1f}" for k in range(N_BINS)])
r_mid = ((R_EDGES[:-1] + R_EDGES[1:]) / 2).numpy()

fig, ax = plt.subplots(figsize=(9, 4))
for i, c in enumerate(CLASSES):
    ax.plot(r_mid, radial_rate.loc[c].to_numpy(), "o-", ms=3, color=f"C{i}", label=c)
ax.set(xlabel="normalized radius r", ylabel="defect rate", title="Radial defect profile by failure type")
ax.legend(fontsize=8, loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.show()
radial_rate.loc[CLASSES]

`Center`는 중심에서, `Edge-Ring`은 가장자리에서, `Donut`은 중간 반지름에서 불량률이 높아요. 패턴마다 **공간적 모양**이 뚜렷해요.
→ PART 4: 이 모양을 **기저함수 회귀**로 매끄러운 함수 $p(r)$, $p(x,y)$ 로 추정해요.

---
# __PART 3. 사례 ① 로트 조기 수율 예측__

### 3-1. 시나리오
> 25장짜리 로트에서 **앞쪽 $K=5$ 장**(웨이퍼 번호 1–5)만 먼저 검사했어요. 그 결과로 **나머지 20장 각각의 수율**을 예측하고 싶어요.

이런 예측이 되면 검사 순서 · 샘플링 계획을 세우거나, 문제 로트를 일찍 찾아내는 데 쓸 수 있어요.

**회귀 문제로 정식화**
- 관측 단위: 웨이퍼 번호 6–25인 웨이퍼 하나 (타깃 $y$ = 그 웨이퍼의 수율)
- 특성 (모두 **같은 로트의 앞 $K$ 장**에서 계산하거나, 검사 전에 알 수 있는 값)

| 특성 | 의미 |
|---|---|
| `first_mean` | 앞 $K$ 장 수율의 평균 |
| `first_std` | 앞 $K$ 장 수율의 표준편차 |
| `first_edge` | 앞 $K$ 장 가장자리($r\ge0.9$) 다이의 불량률 |
| `first_center` | 앞 $K$ 장 중심부($r<0.3$) 다이의 불량률 |
| `log_n_die` | $\log N$ (제품 정보: 웨이퍼당 다이 수) |
| `wafer_index` | 예측할 웨이퍼의 로트 내 번호 |

### 3-2. 데이터셋 만들기
25장이 모두 있는 로트만 써요.

In [ ]:
FEATURES = ["first_mean", "first_std", "first_edge", "first_center", "log_n_die", "wafer_index"]

def build_lot_dataset(wafer, K=5, target_from=None):
    """25장 로트에서 앞 K장(wafer_index ≤ K)으로 특징을 만들고, wafer_index ≥ target_from 인 웨이퍼를 타깃으로 해요."""
    target_from = K + 1 if target_from is None else target_from
    full = wafer[wafer.groupby("lot")["lot"].transform("size") == 25]
    first = full[full["wafer_index"] <= K].assign(
        ctr_def=lambda d: d[["def_r0", "def_r1", "def_r2"]].sum(axis=1),
        ctr_die=lambda d: d[["die_r0", "die_r1", "die_r2"]].sum(axis=1))
    agg = first.groupby("lot").agg(
        first_mean=("yield", "mean"), first_std=("yield", "std"),
        edge_def=("def_r9", "sum"), edge_die=("die_r9", "sum"),
        ctr_def=("ctr_def", "sum"), ctr_die=("ctr_die", "sum"), n_die=("n_die", "first"))
    agg["first_std"] = agg["first_std"].fillna(0.0)                 # K = 1 이면 표준편차가 정의되지 않아요
    agg["first_edge"] = agg["edge_def"] / agg["edge_die"].clip(lower=1)
    agg["first_center"] = agg["ctr_def"] / agg["ctr_die"].clip(lower=1)
    agg["log_n_die"] = agg["n_die"].map(math.log)
    target = full.loc[full["wafer_index"] >= target_from, ["lot", "wafer_index", "yield", "label"]]
    return target.join(agg[FEATURES[:-1]], on="lot").reset_index(drop=True)

lotdata = build_lot_dataset(wafer, K=5)
print(f"로트 {lotdata['lot'].nunique():,}개 × 20장 = {len(lotdata):,}개 관측")
lotdata.head()

### 3-3. 학습/테스트 분할은 **로트 단위**로
같은 로트의 웨이퍼 20장은 **특성이 똑같고 타깃도 서로 강하게 상관**돼 있어요. 웨이퍼를 무작위로 나누면 같은 로트가 학습 · 테스트에 동시에 들어가서 **정보 누수(data leakage)** 가 생겨요.
"처음 보는 로트"에 대한 성능을 재려면 **로트를 단위로** 나눠야 해요 (group split).

표준화 통계(평균 · 표준편차)도 **학습 데이터에서만** 계산해요.

In [ ]:
def lot_split(lots, test_frac=0.2, seed=0):
    """로트 이름 Series → 테스트 로트 여부 (bool 텐서)"""
    uniq = sorted(lots.unique())
    g = torch.Generator().manual_seed(seed)
    perm = torch.randperm(len(uniq), generator=g)[: int(test_frac * len(uniq))]
    test_lots = {uniq[i] for i in perm.tolist()}
    return torch.tensor(lots.isin(test_lots).to_numpy())

def to_tensors(data, features):
    X = torch.tensor(data[features].to_numpy(), dtype=F64)
    y = torch.tensor(data["yield"].to_numpy(), dtype=F64)
    return X, y

is_te = lot_split(lotdata["lot"])
is_tr = ~is_te
X_raw, y_lot = to_tensors(lotdata, FEATURES)
mu, sd = X_raw[is_tr].mean(0), X_raw[is_tr].std(0)
Z_lot = (X_raw - mu) / sd                                       # 학습 통계로 표준화
ones = torch.ones(len(Z_lot), 1, dtype=F64)
D_lot = torch.cat([ones, Z_lot], dim=1)                         # 절편 열 추가

print(f"train {is_tr.sum().item():,}개 ({lotdata.loc[is_tr.numpy(), 'lot'].nunique():,} lots) / "
      f"test {is_te.sum().item():,}개 ({lotdata.loc[is_te.numpy(), 'lot'].nunique():,} lots)")
print("특성 상관행렬 (train):")
pd.DataFrame(torch.corrcoef(Z_lot[is_tr].T).numpy(), index=FEATURES, columns=FEATURES).round(2)

### 3-4. 닫힌 해로 기준 모델 만들기
세 모델을 비교해요.
- **M0** 평균만 예측 (항상 학습 데이터 평균 수율)
- **M1** 단순회귀: `first_mean` 하나만
- **M2** 다중회귀: 특성 6개 전부

In [ ]:
def evaluate(y, y_hat):
    return {"R²": r2_score(y, y_hat).item(), "RMSE": rmse(y, y_hat).item(), "MAE": (y - y_hat).abs().mean().item()}

rows = {}
y_tr, y_te = y_lot[is_tr], y_lot[is_te]
rows["M0 mean only"] = {"train R²": 0.0, **{f"test {k}": v for k, v in evaluate(y_te, torch.full_like(y_te, y_tr.mean().item())).items()}}

cols_M1 = [0, 1]                                     # 절편 + first_mean
fit_M1 = ols(D_lot[is_tr][:, cols_M1], y_tr)
rows["M1 first_mean"] = {"train R²": fit_M1["r2"].item(),
                         **{f"test {k}": v for k, v in evaluate(y_te, D_lot[is_te][:, cols_M1] @ fit_M1["beta"]).items()}}

fit_M2 = ols(D_lot[is_tr], y_tr)
rows["M2 all features"] = {"train R²": fit_M2["r2"].item(),
                           **{f"test {k}": v for k, v in evaluate(y_te, D_lot[is_te] @ fit_M2["beta"]).items()}}
pd.DataFrame(rows).T.round(4)

### 3-5. PyTorch로 학습하기
같은 다중회귀(M2)를 `nn.Linear` + 미니배치 SGD로 학습하고, 닫힌 해와 비교해요. 실제 딥러닝 모델도 **이 루프 구조 그대로** 학습해요.

- **모멘텀**: $\mathbf v_{t+1}=0.9\,\mathbf v_t+\mathbf g_t,\ \ \boldsymbol\beta_{t+1}=\boldsymbol\beta_t-\eta\,\mathbf v_{t+1}$. 지난 기울기를 누적해서, 1-6에서 본 지그재그 방향은 서로 상쇄하고 골짜기 방향으로는 가속해요.
- **코사인 스케줄**: 학습률을 epoch마다 0까지 서서히 줄여서, 미니배치 잡음 때문에 생기는 마지막 흔들림을 없애요.
- 1-6에서 예고한 대로, 특성 6개를 표준화해도 상관이 강해서 헤시안의 조건수가 1보다 훨씬 커요 (아래 출력).

In [ ]:
lam_A = torch.linalg.eigvalsh(2 / is_tr.sum().item() * D_lot[is_tr].T @ D_lot[is_tr])
print(f"표준화한 특성 6개 + 절편: 헤시안 κ = {(lam_A[-1] / lam_A[0]).item():.1f}\n")

torch.manual_seed(0)
train_loader = DataLoader(TensorDataset(Z_lot[is_tr].float(), y_tr.float().unsqueeze(1)),
                          batch_size=1024, shuffle=True, generator=torch.Generator().manual_seed(0))
Z_te_dev = Z_lot[is_te].float().to(device)
y_te_dev = y_te.float().unsqueeze(1).to(device)

model = nn.Linear(len(FEATURES), 1).to(device)
opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=8)   # 학습률을 점점 줄여 마지막 흔들림을 줄여요
loss_fn = nn.MSELoss()

history = []
t0 = time.time()
for epoch in range(1, 9):
    model.train()
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        loss = loss_fn(model(xb), yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item() * len(xb)
    sched.step()
    model.eval()
    with torch.no_grad():
        test_mse = loss_fn(model(Z_te_dev), y_te_dev).item()
    history.append((total / is_tr.sum().item(), test_mse))
    print(f"epoch {epoch}: train MSE {history[-1][0]:.6f}   test MSE {test_mse:.6f}")
print(f"학습 {time.time() - t0:.1f}초 ({device})")

w_nn = torch.cat([model.bias.detach().cpu().double(), model.weight.detach().cpu().double().squeeze(0)])
display(pd.DataFrame({"closed form (lstsq)": fit_M2["beta"].numpy(), "nn.Linear + SGD": w_nn.numpy()},
                     index=["(intercept)"] + FEATURES))

closed_tr = mse(D_lot[is_tr] @ fit_M2["beta"], y_tr).item()
fig, ax = plt.subplots(figsize=(5.5, 3.2))
ax.plot(range(1, 9), [h[0] for h in history], "o-", label="train MSE")
ax.plot(range(1, 9), [h[1] for h in history], "s-", label="test MSE")
ax.axhline(closed_tr, color="k", ls="--", lw=1, label="closed-form train MSE")
ax.set(xlabel="epoch", ylabel="MSE (log)", yscale="log", title="Training nn.Linear on 348k wafers")
ax.legend()
plt.show()

- SGD로 학습한 계수가 닫힌 해와 거의 같아요. 첫 epoch의 학습 MSE가 큰 건 epoch 동안 계속 바뀌는 모델의 손실을 평균했기 때문이에요.
- 테스트 MSE가 학습 MSE보다 조금 작아요 (3-4에서도 테스트 $R^2$ 가 더 높았어요). 과적합의 반대가 아니라 **어떤 로트가 테스트에 뽑혔는지**에 따른 우연이에요. `lot_split` 의 `seed` 를 1–5로 바꾸면 테스트 $R^2$ 는 0.84–0.86 사이에서 바뀌고, 대부분 학습 $R^2$ 보다 작아요.

### 3-6. 평가와 잔차 분석

In [ ]:
with torch.no_grad():
    pred_te = model(Z_te_dev).squeeze(1).double().cpu()
res_te = y_te - pred_te
print({k: round(v, 4) for k, v in evaluate(y_te, pred_te).items()})
print(f"예측값 범위 [{pred_te.min().item():.3f}, {pred_te.max().item():.3f}],  "
      f"0 미만 {(pred_te < 0).sum().item()}개 · 1 초과 {(pred_te > 1).sum().item()}개 (테스트 {len(pred_te):,}개 중)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
hb = axes[0].hexbin(y_te.numpy(), pred_te.numpy(), gridsize=70, bins="log", cmap="viridis", mincnt=1)
axes[0].plot([0, 1], [0, 1], "r--", lw=1)
axes[0].set(xlabel="actual yield", ylabel="predicted yield", title="Predicted vs actual (test lots)")
fig.colorbar(hb, ax=axes[0], label="# wafers (log scale)")
hb = axes[1].hexbin(pred_te.numpy(), res_te.numpy(), gridsize=70, bins="log", cmap="viridis", mincnt=1)
axes[1].axhline(0, color="r", ls="--", lw=1)
axes[1].set(xlabel="predicted yield", ylabel="residual", title="Residual vs fitted")
axes[2].hist(res_te.numpy(), bins=150, density=True, alpha=0.7, label="residuals")
sd_r = res_te.std().item()
gg = torch.linspace(res_te.min().item(), res_te.max().item(), 400, dtype=F64)
axes[2].plot(gg.numpy(), torch.exp(torch.distributions.Normal(0.0, sd_r).log_prob(gg)).numpy(), "r-", lw=1, label="normal (same sd)")
axes[2].set(xlabel="residual", ylabel="density", yscale="log", ylim=(1e-3, 1e2), title="Residual distribution (log)")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

# 불량 패턴별 잔차 크기 (테스트 로트)
res_by_label = (pd.DataFrame({"label": lotdata.loc[is_te.numpy(), "label"].to_numpy(), "|residual|": res_te.abs().numpy()})
                .groupby("label")["|residual|"].agg(["size", "mean", "median"]).reindex(CLASSES + ["unlabeled"]))
res_by_label.round(4)

**관찰**
- 대부분의 웨이퍼는 대각선 근처에 모여 있어요 (테스트 $R^2\approx0.86$).
- **잔차의 분산이 예측값에 따라 달라요** (낮은 수율 구간에서 퍼짐이 커요) → 가정 4(등분산) 위반. 가중최소제곱(WLS)이나 타깃 변환을 생각해 볼 수 있어요.
- 잔차 분포의 꼬리가 같은 표준편차의 정규분포(빨간 선)보다 훨씬 두꺼워요. 패턴별 표를 보면 `Near-full`, `Random`, `Donut` 처럼 **웨이퍼 한 장에 갑자기 대량 불량이 생긴 경우**의 오차가 커요. 같은 로트의 앞 웨이퍼 정보로는 예측하기 어려운 사건이에요.
- 수율은 $[0,1]$ 범위인데 선형모형의 예측은 원리적으로 범위를 벗어날 수 있어요. 이번 테스트에서는 경계를 살짝 넘는 정도지만(위 출력 참고), PART 4의 불량 확률 곡면에서는 0보다 작거나 1보다 큰 값이 확실히 나와요.

> 💬 예측을 $[0,1]$ 안으로 보장하려면 어떻게 할까요? (힌트: $\operatorname{logit}(y)$ 를 회귀하거나, 시그모이드 출력 + 다른 손실)

### 3-7. 계수 해석과 통계적 추론
표준화한 특성의 계수는 **"다른 특성을 고정한 채 그 특성만 1 표준편차 바뀔 때 예측 수율의 변화"** 예요 (편회귀계수). 인과 효과가 아니라 예측 관계예요.

In [ ]:
t_M2 = fit_M2["t"]
coef = pd.DataFrame({"coef (standardized)": fit_M2["beta"].numpy(), "se (iid)": fit_M2["se"].numpy(),
                     "t (iid)": t_M2.numpy()}, index=["(intercept)"] + FEATURES)
coef["effect of +1 sd on yield (%p)"] = 100 * coef["coef (standardized)"]
coef.loc["(intercept)", "effect of +1 sd on yield (%p)"] = float("nan")
coef

- `first_mean` 이 압도적이에요. 다른 특성이 같을 때, 앞 5장의 평균 수율이 1 표준편차(약 9%p) 높으면 나머지 웨이퍼의 예측 수율도 약 9%p 높아요. 거의 1:1이에요.
- 다만 3-3의 상관행렬에서 `first_mean` 과 `first_center` 의 상관이 $-0.93$, `first_edge` 와는 $-0.87$ 이에요. 이렇게 상관이 강하면 "다른 특성을 고정한다"는 해석은 현실에서 거의 일어나지 않는 상황이라 조심해야 해요 (**다중공선성**).
- `first_std` 의 계수가 **양수**예요. `first_mean` 이 같다면 표준편차가 클수록 "앞 5장 중 한두 장만 크게 나빠서 평균이 끌려 내려간" 경우일 가능성이 커요. 그러면 나머지 웨이퍼는 평균보다 좋을 수 있어요. 💬 이 설명이 맞는지 확인하려면 어떤 특성을 넣어 보면 될까요? (힌트: 앞 5장의 중앙값)
- `wafer_index` 는 $|t|\approx5.8$ 로 "유의"하지만 효과는 1 표준편차(약 6장)당 0.04%p예요. 관측이 35만 개나 되면 아주 작은 효과도 유의해져요 → **통계적 유의성 ≠ 실질적 중요성**.

#### (심화) 가정이 깨지면 표준오차도 틀린다: 군집-강건 표준오차
같은 로트의 20개 관측은 특성이 같고 오차도 상관돼 있어서 **가정 4(무상관)가 깨져요**. 이때 $\sigma^2(X^\top X)^{-1}$ 은 분산을 과소평가해요.
로트를 군집 $g$ 로 보는 **샌드위치 추정량**은 오차의 군집 내 상관을 허용해요.

$$\widehat{\operatorname{Cov}}_{CR}(\hat{\boldsymbol\beta})=(X^\top X)^{-1}\Big(\sum_g X_g^\top\mathbf e_g\mathbf e_g^\top X_g\Big)(X^\top X)^{-1}\cdot\frac{G}{G-1}\frac{n-1}{n-p}$$

In [ ]:
Dtr, etr = D_lot[is_tr], y_tr - D_lot[is_tr] @ fit_M2["beta"]
gid = torch.from_numpy(pd.factorize(lotdata.loc[is_tr.numpy(), "lot"])[0])
Gtr, (ntr, ptr) = int(gid.max()) + 1, Dtr.shape
U = torch.zeros(Gtr, ptr, dtype=F64).index_add_(0, gid, Dtr * etr[:, None])     # 로트별 X_gᵀ e_g
bread = torch.linalg.inv(Dtr.T @ Dtr)
V_cr = bread @ (U.T @ U) @ bread * (Gtr / (Gtr - 1)) * ((ntr - 1) / (ntr - ptr))
se_cr = torch.sqrt(torch.diag(V_cr))

pd.DataFrame({"se (iid)": fit_M2["se"].numpy(), "se (cluster-robust)": se_cr.numpy(),
              "ratio": (se_cr / fit_M2["se"]).numpy(),
              "t (iid)": fit_M2["t"].numpy(), "t (cluster)": (fit_M2["beta"] / se_cr).numpy()},
             index=["(intercept)"] + FEATURES)

군집 구조를 고려하면 로트 수준 특성의 표준오차가 몇 배 커져요. 관측이 35만 개여도 이 특성들에 대해 **실질적으로 독립인 정보는 로트 수(약 1.7만 개)에 가깝기** 때문이에요.
단, 로트 안에서 값이 바뀌는 `wafer_index` 는 1.13배로 거의 그대로예요. 모든 로트에 6–25번이 똑같이 들어 있어서 이 계수는 같은 로트 안의 웨이퍼끼리 비교해서 추정되고, 로트 공통의 오차는 서로 상쇄되기 때문이에요.

### 3-8. 몇 장을 검사해야 할까? — 검사 매수 $K$ 와 예측 성능
공정하게 비교하려고 타깃을 **웨이퍼 번호 11–25로 고정**하고, $K=1,\dots,10$ 에 대해 테스트 $R^2$ 를 구해요.

In [ ]:
def lstsq_fit_predict(X_tr, y_tr, X_te):
    """절편을 붙여 최소제곱 → 테스트 예측 (계수가 정확히 결정되지 않는 경우도 lstsq가 처리해요)"""
    A_tr = torch.cat([torch.ones(len(X_tr), 1, dtype=F64), X_tr], 1)
    A_te = torch.cat([torch.ones(len(X_te), 1, dtype=F64), X_te], 1)
    beta = torch.linalg.lstsq(A_tr, y_tr.unsqueeze(1)).solution.squeeze(1)
    return A_te @ beta

K_list = list(range(1, 11))
curve = {"first_mean only": [], "all features": []}
for K in K_list:
    data_K = build_lot_dataset(wafer, K=K, target_from=11)
    te_K = lot_split(data_K["lot"])
    for name, feats_K in [("first_mean only", ["first_mean"]), ("all features", FEATURES)]:
        X_K, y_K = to_tensors(data_K, feats_K)
        pred = lstsq_fit_predict(X_K[~te_K], y_K[~te_K], X_K[te_K])
        curve[name].append(r2_score(y_K[te_K], pred).item())

fig, ax = plt.subplots(figsize=(6, 3.6))
for name, vals in curve.items():
    ax.plot(K_list, vals, "o-", label=name)
ax.set(xlabel="# inspected wafers K (index 1..K)", ylabel="test R² (targets: index 11–25)", title="Inspection budget vs. prediction quality")
ax.legend()
plt.show()

# 천장: 타깃 웨이퍼들(11–25번)의 '실제 로트 평균'을 정확히 안다면? (로트 수준 정보로 얻을 수 있는 최대치)
y_c = torch.tensor(data_K["yield"].to_numpy(), dtype=F64)
gid_c = torch.tensor(pd.factorize(data_K["lot"])[0])
G_c = int(gid_c.max()) + 1
lot_avg = (torch.zeros(G_c, dtype=F64).index_add_(0, gid_c, y_c)
           / torch.zeros(G_c, dtype=F64).index_add_(0, gid_c, torch.ones_like(y_c)))
print(f"천장 (실제 로트 평균으로 예측): test R² = {r2_score(y_c[te_K], lot_avg[gid_c][te_K]).item():.3f}")
pd.DataFrame(curve, index=pd.Index(K_list, name="K")).T.round(4)

- 1장만 검사해도 $R^2\approx0.77$, 5장이면 0.86 안팎이에요. 그 뒤로는 5장을 더 검사해도 0.01–0.02 정도만 늘어요. **수확체감**이 뚜렷해요.
- 왜 0.87 근처에서 멈출까요? 특성이 (`wafer_index` 만 빼면) 모두 **로트 수준**이라, 같은 로트 안 웨이퍼끼리의 차이는 설명할 수 없어요. 타깃 웨이퍼들의 로트 평균을 정확히 안다고 해도 $R^2$ 는 위 출력의 '천장' 정도예요 (2-8의 $\eta^2$ 와 같은 계산). 지금 모형은 이 천장에 꽤 가까워요.
- 비용(검사 시간) 함수를 정의하면 "최적 $K$" 를 정할 수 있어요 → 과제 3.

---
# __PART 4. 사례 ② 웨이퍼 맵 공간 회귀 — 기저함수 · 과적합 · 릿지__

### 4-1. 모형
다이 하나를 관측 단위로 봐요. 정규화 좌표 $(x,y)$ 에 있는 다이의 불량 여부 $z\in\{0,1\}$ 에 대해

$$\mathbb E[z\mid x,y]=P(z=1\mid x,y)=:p(x,y)\approx\beta_0+\boldsymbol\phi(x,y)^\top\mathbf w$$

를 최소제곱으로 추정해요 (**선형확률모형**). $\mathbb E[z\mid x,y]$ 는 $L^2$ 에서 $z$ 를 가장 잘 근사하는 $(x,y)$ 의 함수라서, 최소제곱은 그 함수를 기저 $\boldsymbol\phi$ 가 만드는 공간에 **사영**한 것을 추정해요.

- 기저함수로는 다항식 $\boldsymbol\phi(x,y)=\big(x^ay^b\big)_{1\le a+b\le d}$ 를 써요. 특성 수는 $q=\frac{(d+1)(d+2)}2-1$ 개예요.
- 한계: $\hat p$ 가 $[0,1]$ 을 벗어날 수 있고, 분산 $p(1-p)$ 가 위치마다 달라요 (이분산). → 다음 시간의 **로지스틱 회귀**가 이 문제를 해결해요.

### 4-2. 패턴별 반경 프로파일: 1.8억 개 다이에 대한 정확한 OLS
반지름만의 함수 $p(r)=\sum_{k=0}^{6}\beta_kr^k$ 를 패턴별로 추정해요. 라벨이 있는 웨이퍼의 다이는 **약 1.8억 개**나 돼요.
하지만 OLS 해는 두 요약량 $\Phi^\top\Phi\ (7\times7)$ 와 $\Phi^\top\mathbf z\ (7)$ 만으로 정해져요 (가우시안 선형모형에서는 $\mathbf z^\top\mathbf z$ 와 함께 **충분통계량**이에요).

$$\Phi^\top\Phi=\sum_{\text{wafers}}\sum_{\text{dies}}\boldsymbol\phi(r)\boldsymbol\phi(r)^\top=\sum_{\text{pixels}}D_{\text{pixel}}\,\boldsymbol\phi(r_{\text{pixel}})\boldsymbol\phi(r_{\text{pixel}})^\top,\qquad
\Phi^\top\mathbf z=\sum_{\text{pixels}}Z_{\text{pixel}}\,\boldsymbol\phi(r_{\text{pixel}})$$

여기서 $D_{\text{pixel}}$ 은 그 픽셀 위치가 다이인 웨이퍼 수, $Z_{\text{pixel}}$ 은 그 위치가 불량인 웨이퍼 수예요. 같은 크기의 맵끼리 픽셀별로 더해 두면 **설계행렬을 만들지 않고도 정확한 해**를 얻어요. 데이터가 메모리보다 커도 이렇게 스트리밍으로 회귀할 수 있어요.

**(심화) 조건수 이야기**: 단항식 $r^k$ 는 $[0,1]$ 에서 서로 거의 평행해서 $\Phi^\top\Phi$ 의 조건수가 매우 커요. 반면 $s=2r^2-1$ 로 두고 르장드르 다항식 $P_k(s)$ 를 쓰면(= 회전대칭 **Zernike 다항식** $R_{2k}^0(r)$, $k=0,\dots,m$), 면적요소가 $dA=2\pi r\,dr=\frac\pi2\,ds$ 라서

$$\int_{\text{unit disk}}P_j(2r^2-1)\,P_k(2r^2-1)\,dA=\frac\pi2\int_{-1}^{1}P_j(s)P_k(s)\,ds=\frac{\pi}{2k+1}\,\delta_{jk}$$

예요. 다이가 원판 위에 고르게 퍼져 있다면 Gram 행렬이 대각행렬 $\operatorname{diag}(\pi/(2k+1))$ 에 비례해서 조건수가 정확히 $2m+1$ ($m=6$ 이면 13) 이 돼요.
단, 이 기저가 만드는 공간은 $\{1,r^2,\dots,r^{12}\}$ 라서 위에서 적합하는 $\{1,r,\dots,r^{6}\}$ 과 **다른 함수공간**이에요. 그래서 공정한 비교 상대는 같은 공간을 만드는 짝수 차수 단항식 $\{1,r^2,\dots,r^{12}\}$ 예요 (균등한 원판에서 이 Gram 행렬은 $\pi\times$(7차 힐베르트 행렬)이라 조건수가 약 $4.8\times10^8$).
아래 코드에서 세 기저의 조건수를 함께 계산해요. 적합은 단항식 $\{1,\dots,r^6\}$ 으로 하고, Zernike 적합은 과제 2로 남겨요.

여기서는 $7\times7$ 정규방정식을 `solve` 로 바로 풀어요. $\kappa\approx10^9$ 이어도 `float64`($\varepsilon\approx10^{-16}$)에서는 상대오차가 대략 $\kappa\varepsilon\approx10^{-7}$ 수준이라 괜찮지만, 1-3에서 말한 조건수 문제가 실제로 나타나는 예예요.

In [ ]:
DEG_R, N_FINE = 6, 40

def radial_basis(r, d=DEG_R):
    """단항식 [1, r, r², …, r^d]"""
    return torch.stack([r ** k for k in range(d + 1)], dim=-1)

def zernike_radial(r, deg=DEG_R):
    """회전대칭 Zernike 다항식 R_{2k}^0(r) = P_k(2r² − 1), k = 0 … deg (르장드르 점화식)"""
    s = 2 * r ** 2 - 1
    P = [torch.ones_like(s), s]
    for k in range(1, deg):
        P.append(((2 * k + 1) * s * P[k] - k * P[k - 1]) / (k + 1))
    return torch.stack(P[: deg + 1], dim=-1)

XtX = {c: torch.zeros(DEG_R + 1, DEG_R + 1, dtype=F64) for c in CLASSES}
Xtz = {c: torch.zeros(DEG_R + 1, dtype=F64) for c in CLASSES}
fine_D = {c: torch.zeros(N_FINE, dtype=F64) for c in CLASSES}      # 확인용: 촘촘한 반지름 구간별 다이 수
fine_Z = {c: torch.zeros(N_FINE, dtype=F64) for c in CLASSES}      #          … 불량 수
G_zern = torch.zeros(DEG_R + 1, DEG_R + 1, dtype=F64)              # Gram 행렬 (조건수 비교용): Zernike 기저
G_even = torch.zeros(DEG_R + 1, DEG_R + 1, dtype=F64)              #                           짝수 차수 단항식 [1, r², …, r¹²]
LABELS = df["label"].to_numpy()

t0 = time.time()
labeled = df[df["label"] != "unlabeled"]
for (H, W), pos in labeled.groupby(["H", "W"]).indices.items():
    idx = labeled.index.to_numpy()[pos]
    maps = stack_maps(idx)
    xg_, yg_ = normalized_grid(H, W)
    r = torch.sqrt(xg_**2 + yg_**2).reshape(-1)                     # 픽셀별 반지름
    Phi, Psi = radial_basis(r), zernike_radial(r)                    # (픽셀 수, 7)
    fbin = (r.clamp(max=1 - 1e-9) * N_FINE).long()
    D_all = (maps > 0).sum(0).reshape(-1).to(F64)
    G_zern += Psi.T @ (D_all[:, None] * Psi)
    Pe = torch.stack([r ** (2 * k) for k in range(DEG_R + 1)], dim=-1)
    G_even += Pe.T @ (D_all[:, None] * Pe)
    labs = LABELS[idx]
    for c in set(labs):
        sel = maps[torch.from_numpy(labs == c)]
        D = (sel > 0).sum(0).reshape(-1).to(F64)        # 픽셀별: 다이인 웨이퍼 수
        Zc = (sel == 2).sum(0).reshape(-1).to(F64)      # 픽셀별: 불량인 웨이퍼 수
        XtX[c] += Phi.T @ (D[:, None] * Phi)
        Xtz[c] += Phi.T @ Zc
        fine_D[c] += torch.bincount(fbin, weights=D, minlength=N_FINE)
        fine_Z[c] += torch.bincount(fbin, weights=Zc, minlength=N_FINE)
beta_r = {c: torch.linalg.solve(XtX[c], Xtz[c]) for c in CLASSES}
n_obs = {c: XtX[c][0, 0].item() for c in CLASSES}                  # [ΦᵀΦ]₀₀ = Σ 1 = 다이 수
G_mono = sum(XtX.values())
print(f"{time.time() - t0:.1f}초, 전체 다이 수 = {sum(n_obs.values()):,.0f}")
print(f"κ(ΦᵀΦ): [1, r, …, r⁶] {torch.linalg.cond(G_mono).item():.2e}   |   같은 공간 비교: [1, r², …, r¹²] "
      f"{torch.linalg.cond(G_even).item():.2e}  vs  Zernike {torch.linalg.cond(G_zern).item():.1f}  (균등분포면 {2 * DEG_R + 1})")

rg = torch.linspace(0, 1, 200, dtype=F64)
r_fine = (torch.arange(N_FINE, dtype=F64) + 0.5) / N_FINE
fig, axes = plt.subplots(3, 3, figsize=(12, 8), sharex=True)
for ax, (i, c) in zip(axes.flat, enumerate(CLASSES)):
    ax.plot(r_fine.numpy(), (fine_Z[c] / fine_D[c].clamp_min(1)).numpy(), "o", ms=3, color="k", label=f"empirical rate ({N_FINE} bins)")
    ax.plot(rg.numpy(), (radial_basis(rg) @ beta_r[c]).numpy(), color=f"C{i}", lw=2, label=f"OLS, degree {DEG_R}")
    ax.set_title(f"{c}  ({n_obs[c] / 1e6:.1f}M dies)", fontsize=10)
    ax.set_ylim(bottom=0)
for ax in axes[-1]:
    ax.set_xlabel("r")
axes[0, 0].legend(fontsize=7)
plt.suptitle(r"Radial defect probability $\hat p(r)$ fitted on all labeled dies", y=1.0)
plt.tight_layout()
plt.show()

- 대부분의 패턴에서 촘촘한 구간의 경험적 불량률(점)을 다항식(선)이 잘 따라가요. `Edge-Ring`은 $r\to1$ 에서 급격히 올라가요.
- 하지만 `Donut` 처럼 모양이 복잡하면 6차 다항식으로는 봉우리 높이와 $r\approx0.1$ 근처의 골을 다 따라가지 못해요 (**기저의 편향**).
- $r\to0$ 근처에서는 곡선이 크게 흔들려요 (`Donut` $\hat p(0)\approx1$, `Center` 는 $\hat p(0)>1$). 반지름 $r$ 근처의 다이 수는 $r$ 에 비례해서(원 둘레 $2\pi r$) 중심에는 데이터가 아주 적고, 그래서 그 근처의 추정 분산이 커요. $\hat p$ 가 1을 넘는 건 4-1에서 말한 **선형확률모형의 한계**예요.
- 또 **정중앙($r<0.05$)에서는 모든 패턴의 불량률이 튀어요** (`none` 조차!). 패턴 라벨과 무관한 데이터의 특이점이에요. 아래 (심화) 셀에서 확인해요.

> 💬 `Loc`, `Edge-Loc`, `Scratch` 처럼 **패턴 위치가 웨이퍼마다 다른** 경우는 반지름 프로파일로 잘 요약될까요? 패턴별로 평균을 내면 위치가 섞여서 지워져요. 그래서 4-3부터는 웨이퍼 **한 장**에 2차원 곡면 $p(x,y)$ 를 맞춰요. 먼저 모양이 분명한 `Edge-Ring` 한 장으로 차수 선택과 릿지를 익히고, 4-5에서 `Loc`, `Edge-Loc`, `Scratch` 에 적용해 봐요.

#### (심화) 데이터를 의심하기: 정중앙 다이
맵 크기 $H, W$ 가 모두 홀수이면 정중앙에 다이가 하나 있어요. 그 다이, 바로 옆 8개 다이, 그리고 **같은 맵들 전체**의 불량률을 비교해요.

In [ ]:
c_die = c_bad = nb_die = nb_bad = oo_die = oo_bad = 0
for (H, W), idx in df.groupby(["H", "W"]).indices.items():
    if H % 2 == 0 or W % 2 == 0:
        continue
    maps = stack_maps(idx)
    ci, cj = H // 2, W // 2
    center = maps[:, ci, cj]
    ring8 = maps[:, ci - 1:ci + 2, cj - 1:cj + 2].reshape(len(idx), 9)[:, [0, 1, 2, 3, 5, 6, 7, 8]]
    c_die += (center > 0).sum().item();  c_bad += (center == 2).sum().item()
    nb_die += (ring8 > 0).sum().item();  nb_bad += (ring8 == 2).sum().item()
    oo_die += (maps > 0).sum().item();   oo_bad += (maps == 2).sum().item()
print(f"정중앙 다이 불량률             {c_bad / c_die:.3f}   (다이 {c_die:,}개)")
print(f"주변 8개 다이 불량률           {nb_bad / nb_die:.3f}   (다이 {nb_die:,}개)")
print(f"같은 맵들(H, W 모두 홀수) 전체 {oo_bad / oo_die:.3f}   (다이 {oo_die:,}개)")

정중앙 다이의 불량률은 바로 옆 8개의 약 3배, 같은 맵 전체의 약 7배예요. 옆 8개도 전체의 2배가 넘어서, **중심으로 갈수록 불량률이 올라가다가 정중앙 한 칸에서 크게 튀는** 모양이에요.
측정 · 테스트 과정의 특수 위치(예: 정렬용 위치)일 수도 있고, 데이터 생성 과정의 특성일 수도 있어요. 원인은 데이터만으로는 알 수 없지만, **모형을 맞추기 전에 이런 특이점을 찾아내는 것**이 데이터 분석의 중요한 부분이에요.

### 4-3. 웨이퍼 한 장의 불량 확률 곡면과 차수 선택 (교차검증)
`Edge-Ring` 웨이퍼 한 장(다이 600개)에 2차원 다항식 $\hat p(x,y)$ 를 맞춰요. 다이를 무작위로 5개 폴드로 나눠 **5-fold 교차검증 × 2회 반복**으로 차수 $d$ 를 골라요.

$$\text{CV}(d)=\frac1{2\cdot5}\sum_{m=1}^{2}\sum_{f=1}^{5}\frac{1}{|\mathcal V_{m,f}|}\sum_{i\in\mathcal V_{m,f}}\big(z_i-\hat p^{(-m,f)}_d(x_i,y_i)\big)^2$$

($m$ 은 반복 번호예요. 반복할 때마다 폴드를 새로 무작위로 나눠요. $\hat p^{(-m,f)}_d$ 는 폴드 $\mathcal V_{m,f}$ 를 빼고 학습한 모형이에요.)

In [ ]:
def poly_features(x, y, d):
    """x^a y^b (1 ≤ a+b ≤ d), 총차수 순서. 상수항은 제외 (절편은 따로)"""
    cols = [x ** (k - j) * y ** j for k in range(1, d + 1) for j in range(k + 1)]
    return torch.stack(cols, dim=1) if cols else torch.zeros(len(x), 0, dtype=x.dtype)

def n_poly(d):
    return (d + 1) * (d + 2) // 2 - 1

def wafer_xyz(wm):
    """웨이퍼 맵 → 다이별 (x, y, z) 와 이미지로 되돌릴 위치 정보"""
    H, W = wm.shape
    t = torch.from_numpy(wm).long()
    ii, jj = torch.nonzero(t > 0, as_tuple=True)
    x = (jj.to(F64) - (W - 1) / 2) / (W / 2)
    y = ((H - 1) / 2 - ii.to(F64)) / (H / 2)
    return x, y, (t[ii, jj] == 2).to(F64), (ii, jj, H, W)

def to_image(values, where):
    ii, jj, H, W = where
    img = torch.full((H, W), float("nan"), dtype=F64)
    img[ii, jj] = values
    return img

def kfold_ids(n, k=5, seed=0):
    g = torch.Generator().manual_seed(seed)
    fold = torch.empty(n, dtype=torch.long)
    fold[torch.randperm(n, generator=g)] = torch.arange(n) % k
    return fold

def cv_mse(P, z, lam=0.0, k=5, reps=2):
    total = 0.0
    for rep in range(reps):
        fold = kfold_ids(len(z), k, seed=rep)
        for f in range(k):
            tr_, va_ = fold != f, fold == f
            m = ridge_fit(P[tr_], z[tr_], lam)
            total += mse(ridge_predict(m, P[va_]), z[va_]).item()
    return total / (k * reps)

EX_IDX = 810328                                     # 예시 웨이퍼 (Edge-Ring, 다이 600개)
wm_ex = df.at[EX_IDX, "waferMap"]
print(f"웨이퍼 #{EX_IDX}: {df.at[EX_IDX, 'lot']} / {df.at[EX_IDX, 'wafer_index']}번, "
      f"label = {df.at[EX_IDX, 'label']}, 맵 {wm_ex.shape[0]}×{wm_ex.shape[1]}")
x_ex, y_ex, z_ex, where_ex = wafer_xyz(wm_ex)

D_MAX = 16
P_all = poly_features(x_ex, y_ex, D_MAX)            # 차수 d 의 특성 = 앞쪽 n_poly(d) 개 열
deg_list = list(range(D_MAX + 1))
cv_d = [cv_mse(P_all[:, :n_poly(d)], z_ex) for d in deg_list]
tr_d = [mse(ridge_predict(ridge_fit(P_all[:, :n_poly(d)], z_ex), P_all[:, :n_poly(d)]), z_ex).item() for d in deg_list]
d_best = deg_list[int(torch.tensor(cv_d).argmin())]
print(f"다이 {len(z_ex)}개, 수율 {1 - z_ex.mean().item():.3f},  CV로 고른 차수 d = {d_best} (특성 {n_poly(d_best)}개)")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(deg_list, tr_d, "o-", label="train MSE (all dies)")
ax.plot(deg_list, cv_d, "s-", label="5-fold CV MSE")
ax.axvline(d_best, color="k", ls=":", lw=1)
ax.set(xlabel="polynomial degree d", ylabel="MSE", title="Model selection for one wafer")
ax.legend()
plt.show()

In [ ]:
PROB_CMAP = plt.get_cmap("magma_r").copy()
PROB_CMAP.set_bad("white")                          # 웨이퍼 밖(NaN)은 흰색

def show_prob(ax, img, title):
    """추정한 불량 확률 지도 (표시할 때만 [0, 1]로 자르고, 웨이퍼 밖 NaN은 흰색)"""
    im = ax.imshow(img.clamp(0, 1).numpy(), cmap=PROB_CMAP, vmin=0, vmax=1, interpolation="nearest",
                   extent=(-1, 1, -1, 1))
    ax.set_aspect("equal")
    ax.set_axis_off()
    ax.set_title(title, fontsize=9)
    return im

shown = [1, 2, d_best, D_MAX]
fig, axes = plt.subplots(1, len(shown) + 1, figsize=(3.0 * (len(shown) + 1), 3.2))
show_wafer(axes[0], wm_ex, "original (Edge-Ring)")
for ax, d in zip(axes[1:], shown):
    m = ridge_fit(P_all[:, :n_poly(d)], z_ex)
    p_hat = ridge_predict(m, P_all[:, :n_poly(d)])
    im = show_prob(ax, to_image(p_hat, where_ex), f"d = {d} ({n_poly(d)} features)\nCV MSE {cv_d[d]:.4f}")
fig.colorbar(im, ax=axes, shrink=0.8, label=r"$\hat p(x, y)$ (clipped to [0, 1])")
plt.show()

# 그림은 다이 위치에서만 그려요. 다이가 없는 곳(다이 사이 · 가장자리 바깥)까지 촘촘한 격자에서 p̂ 를 계산해 보면?
gx, gy = torch.meshgrid(torch.linspace(-1, 1, 201, dtype=F64), torch.linspace(-1, 1, 201, dtype=F64), indexing="ij")
inside = gx**2 + gy**2 <= 1
for d in [d_best, D_MAX]:
    m = ridge_fit(P_all[:, :n_poly(d)], z_ex)
    p_die = ridge_predict(m, P_all[:, :n_poly(d)])
    p_grid = ridge_predict(m, poly_features(gx[inside], gy[inside], d))
    print(f"d = {d:2d}: 다이 위치 p̂ ∈ [{p_die.min().item():5.2f}, {p_die.max().item():5.2f}]   "
          f"원판 전체(다이 사이 포함) p̂ ∈ [{p_grid.min().item():5.2f}, {p_grid.max().item():5.2f}]")

- $d=1$: 기울어진 평면이라 링을 전혀 표현하지 못해요.
- $d=2$: 그릇 모양의 2차 곡면($x^2+y^2$ 같은)이라 "가장자리가 높다"는 경향은 잡지만, 가장자리 두세 줄에 몰린 **좁은 링**은 표현하지 못해요. 그래서 가장자리 확률은 낮게, 중심은 0 아래로 추정해요 (**과소적합**, 편향 큼).
- $d$ = CV 최적: 가장자리 링이 매끄럽게 드러나요.
- $d=16$: 특성 152개로 다이 하나하나의 잡음까지 따라가서 곡면이 얼룩덜룩해요 (**과적합**, 분산 큼). 그림은 다이 위치에서만 그려서 잘 안 보이지만, **웨이퍼 가장자리 근처의 다이 사이 · 바깥**(데이터가 없는 곳)에서 계산해 보면 $\hat p$ 가 $[0,1]$ 을 크게 벗어나요 (위 출력). 다항식은 데이터 경계에서 가장 크게 흔들려요 (1-7의 1차원 예와 같은 현상).

### 4-4. 릿지 정규화: 차수 대신 벌점으로 복잡도 조절
$d=16$ 을 고정하고 $\lambda$ 만 바꿔요. 1-7에서 본 유효 자유도 $\operatorname{df}(\lambda)=\sum_j d_j^2/(d_j^2+n\lambda)$ 도 함께 그려요.

In [ ]:
lam_grid = torch.logspace(-8, 3, 23, dtype=F64)
cv_l = [cv_mse(P_all, z_ex, lam=l.item()) for l in lam_grid]
lam_star = lam_grid[int(torch.tensor(cv_l).argmin())].item()

Z_ex = (P_all - P_all.mean(0)) / P_all.std(0)
sv2 = torch.linalg.svdvals(Z_ex) ** 2
dof = [(sv2 / (sv2 + len(z_ex) * l)).sum().item() for l in lam_grid]
print(f"CV로 고른 λ = {lam_star:.1e},  유효 자유도 df(λ) = {(sv2 / (sv2 + len(z_ex) * lam_star)).sum().item():.1f} (특성 {P_all.shape[1]}개)")
print(f"λ = 1e-8 에서도 df = {dof[0]:.1f}  (rank(Z) = {torch.linalg.matrix_rank(Z_ex).item()})")
print(f"CV MSE: 차수 선택 최적 {min(cv_d):.4f}  vs  릿지 최적 {min(cv_l):.4f}")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.semilogx(lam_grid.numpy(), cv_l, "s-", color="C1", label="5-fold CV MSE")
ax.axvline(lam_star, color="k", ls=":", lw=1)
ax.set(xlabel=r"$\lambda$", ylabel="CV MSE", title=f"Ridge, d = {D_MAX}")
ax2 = ax.twinx()
ax2.semilogx(lam_grid.numpy(), dof, "--", color="C2", label=r"df($\lambda$)")
ax2.set_ylabel(r"effective dof df($\lambda$)", color="C2")
ax2.grid(False)
h1, l1 = ax.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax.legend(h1 + h2, l1 + l2, loc="upper center")
plt.show()

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
show_wafer(axes[0], wm_ex, "original")
for ax, lam in zip(axes[1:], [1e-8, lam_star, 1e2]):
    m = ridge_fit(P_all, z_ex, lam)
    im = show_prob(ax, to_image(ridge_predict(m, P_all), where_ex), f"d = {D_MAX}, λ = {lam:.0e}\n‖w‖ = {m['w'].norm().item():.2f}")
fig.colorbar(im, ax=axes, shrink=0.8, label=r"$\hat p(x, y)$")
plt.show()

- CV가 고른 $\lambda$ 에서 유효 자유도는 약 19예요. 특성은 152개인데, 실제로는 차수 6 모형(특성 27개)보다도 단순한 모형처럼 행동해요.
- CV MSE는 차수 선택과 릿지가 거의 같아요. 릿지의 장점은 성능이 더 좋다는 게 아니라, 복잡도를 **연속적인 손잡이 하나($\lambda$)** 로 조절한다는 데 있어요. 특성에 차수 같은 자연스러운 순서가 없을 때 특히 유용해요.
- $\lambda$ 가 너무 작으면 $d=16$ OLS처럼 잡음을 따라가고($\lVert\mathbf w\rVert$ 가 큼), 너무 크면 계수가 모두 0으로 줄어 평균 불량률만 남아요 (과소적합).
- $\lambda=10^{-8}$ 에서도 df가 152가 아니라 약 130인 이유: 고차 단항식 특성끼리 거의 공선이라, $d_j^2$ 가 $n\lambda=600\times10^{-8}$ 보다 작은 방향이 많기 때문이에요 (1-7의 $\operatorname{df}(\lambda)$ 식).

### 4-5. 패턴별 공간 회귀 갤러리
패턴마다 웨이퍼 한 장씩 골라 $d=12$ 다항식 + CV로 고른 $\lambda$ 로 불량 확률 곡면을 추정해요. (패턴이 잘 보이는 웨이퍼를 미리 골라 뒀어요. `GALLERY`의 번호를 바꿔서 다른 웨이퍼도 해 보세요.)
웨이퍼 한 장이 **회귀계수 $\hat{\mathbf w}\in\mathbb R^{90}$** 로 요약돼요. 다만 여기서는 웨이퍼마다 자기 다이 배치로 특성을 표준화하고 $\lambda$ 도 따로 골라서, $\hat{\mathbf w}$ 를 웨이퍼끼리 그대로 비교할 수는 없어요. 패턴 분류의 특징으로 쓰려면 모든 웨이퍼에 **같은 표준화 기준과 같은 $\lambda$** (또는 Zernike처럼 원판에서 직교하는 공통 기저)를 써야 해요.

In [ ]:
GALLERY = {"none": 649384, "Center": 394506, "Donut": 279757, "Edge-Loc": 648683, "Edge-Ring": 385104,
           "Loc": 220247, "Near-full": 255069, "Random": 736832, "Scratch": 22044}
D_GAL, LAMS_GAL = 12, [1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0]

fig, axes = plt.subplots(2, len(GALLERY), figsize=(1.9 * len(GALLERY), 4.6))
for j, (c, k) in enumerate(GALLERY.items()):
    assert df.at[k, "label"] == c
    x_k, y_k, z_k, where_k = wafer_xyz(df.at[k, "waferMap"])
    P_k = poly_features(x_k, y_k, D_GAL)
    lam_k = min(LAMS_GAL, key=lambda l: cv_mse(P_k, z_k, lam=l, reps=1))
    p_k = ridge_predict(ridge_fit(P_k, z_k, lam_k), P_k)
    show_wafer(axes[0, j], df.at[k, "waferMap"], f"{c}\n#{k}")
    im = show_prob(axes[1, j], to_image(p_k, where_k), f"λ = {lam_k:.0e}")
fig.colorbar(im, ax=axes[1], shrink=0.8, label=r"$\hat p$")
plt.suptitle("Top: wafer map · Bottom: ridge-regularized polynomial surface (d = 12)", y=1.02)
plt.show()

- `Center`, `Donut`, `Edge-Ring`, `Edge-Loc`, `Loc`, `Near-full`: 매끄러운 곡면이 패턴의 위치와 모양을 잘 잡아요 (작은 $\lambda$).
- `none`: CV가 큰 $\lambda$ 를 골라 곡면이 거의 평평해요. `Random` 도 비교적 큰 $\lambda$ 가 골라져 곡면이 밋밋해요. "뚜렷한 공간 구조가 거의 없다"는 것을 회귀가 스스로 알려 주는 셈이에요.
- `Scratch`: 비교적 큰 $\lambda$ 가 골라져 선이 거의 지워졌어요. 가늘고 긴 선을 매끄러운 **전역 다항식(차수 12)** 으로 표현하려면 차수가 아주 높아야 하고, 그만큼 분산이 커지기 때문이에요 (**기저의 한계**). 즉 큰 $\lambda$ 가 항상 "구조가 없다"는 뜻은 아니에요. 국소적인 패턴에는 다른 기저나 학습하는 특징(CNN)이 필요해요.